# PRISM Uplift Modeling Workflow — GenRocket 10k Dataset
Migrated from `Uplift Model Code_rh06032026.ipynb` to run on `Genrocket_10k_seed1_updated.csv`.

**Outcome:** `outcome_ed_90d`  
**Treatment:** `intervention_flag`


---
## 1. Imports & Setup
---


In [ ]:
import sys, warnings, os
from pathlib import Path
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from sklearn.linear_model import LogisticRegressionCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.exceptions import ConvergenceWarning
import xgboost as xgb
try:
    import shap
except ImportError:
    shap = None

for candidate in [Path.cwd(), Path.cwd() / 'Code', Path.cwd().parent / 'Code']:
    if (candidate / '_prism_model_utils.py').exists():
        sys.path.insert(0, str(candidate))
        break

from _prism_model_utils import (
    project_root, ensure_output_folder, split_train_test,
    ntile_desc, make_design_matrix, clean_feature_names,
    resolve_xgb_gpu_params, shap_importance_frame,
    xgb_importance_frame, xgb_training_params,
    assert_xgb_booster_uses_cuda, impute_numeric,
    impute_categorical, to_binary, align_to_columns,
    require_columns, present_columns,
)

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
PROJECT_ROOT = project_root()
SEED = 123
np.random.seed(SEED)


---
## 2. Configuration & Output Setup
---


In [ ]:
REQUIRE_GPU_FOR_XGBOOST = True
RUN_CPU_ONLY_COMPARISON_MODELS = True
XGBOOST_CUDA_DEVICE = 0
XGB_GPU_PARAMS = resolve_xgb_gpu_params(cuda_device=XGBOOST_CUDA_DEVICE) if REQUIRE_GPU_FOR_XGBOOST else {}

output_folder = ensure_output_folder(PROJECT_ROOT / 'Outputs' / 'upliftGenrocket' / 'Python')
tlearner_folder = ensure_output_folder(output_folder / 'T-Learner')
xgboost_output_folder = ensure_output_folder(tlearner_folder / 'XGBoost')
glmnet_output_folder = ensure_output_folder(tlearner_folder / 'GLMNet')
predictor_dist_folder = ensure_output_folder(output_folder / 'Predictor_Distributions')

xgboost_output_path = xgboost_output_folder / 'uplift_scored_output.csv'
xgboost_summary_path = xgboost_output_folder / 'uplift_decile_summary.csv'
glmnet_output_path = glmnet_output_folder / 'uplift_scored_output.csv'
glmnet_summary_path = glmnet_output_folder / 'uplift_decile_summary.csv'

output_path = xgboost_output_path
summary_path = xgboost_summary_path

print('Project root:', PROJECT_ROOT)
print('Output root:', output_folder)
print('XGBoost outputs:', xgboost_output_folder)
print('GLMNet outputs:', glmnet_output_folder)
print('Predictor distributions:', predictor_dist_folder)
print('XGBoost GPU params:', XGB_GPU_PARAMS)


---
## 3. Helper Functions
---


In [ ]:
def safe_auc(y_true, y_pred, label):
    y_series = pd.Series(y_true).dropna()
    if y_series.nunique() < 2:
        print(f'{label} AUC: cannot calculate because only one outcome class is present')
        return np.nan
    auc_value = roc_auc_score(y_true, y_pred)
    print(f'{label} AUC: {auc_value:.4f}')
    return auc_value


def make_dmatrix(x_matrix, y=None):
    if y is None:
        return xgb.DMatrix(x_matrix, feature_names=list(x_matrix.columns))
    return xgb.DMatrix(x_matrix, label=np.asarray(y, dtype=float), feature_names=list(x_matrix.columns))


def fit_xgb_cv_grid(x_matrix, y, grid, nrounds_max=500, nfold=5, seed=123):
    y_array = np.asarray(y, dtype=float)
    dtrain = make_dmatrix(x_matrix, y_array)
    class_counts = pd.Series(y_array).value_counts()
    folds = int(min(nfold, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two outcome classes with at least two rows each for XGBoost CV.')

    results = []
    best_model_info = None
    best_auc = -np.inf

    for params_grid in grid:
        params_i = xgb_training_params(
            XGB_GPU_PARAMS,
            {
                'max_depth': params_grid['max_depth'],
                'eta': params_grid['eta'],
                'min_child_weight': params_grid['min_child_weight'],
                'subsample': 0.8,
                'colsample_bytree': 0.8,
            },
            eval_metric='auc',
            seed=seed,
        )
        cv_i = xgb.cv(
            params=params_i,
            dtrain=dtrain,
            num_boost_round=nrounds_max,
            nfold=folds,
            stratified=True,
            early_stopping_rounds=20,
            seed=seed,
            verbose_eval=False,
        )
        auc_column = 'test-auc-mean'
        best_iter_i = int(cv_i[auc_column].idxmax())
        best_auc_i = float(cv_i.loc[best_iter_i, auc_column])
        best_nrounds_i = best_iter_i + 1
        results.append({
            'max_depth': params_grid['max_depth'],
            'eta': params_grid['eta'],
            'min_child_weight': params_grid['min_child_weight'],
            'best_nrounds': best_nrounds_i,
            'cv_auc': best_auc_i,
        })
        if best_auc_i > best_auc:
            best_auc = best_auc_i
            best_model_info = {'params': params_i, 'best_nrounds': best_nrounds_i, 'cv_auc': best_auc_i}

    search_results = pd.DataFrame(results).sort_values('cv_auc', ascending=False).reset_index(drop=True)
    final_model = xgb.train(
        params=best_model_info['params'],
        dtrain=dtrain,
        num_boost_round=best_model_info['best_nrounds'],
        verbose_eval=False,
    )
    return {
        'model': final_model,
        'best_params': best_model_info['params'],
        'best_nrounds': best_model_info['best_nrounds'],
        'best_cv_auc': best_model_info['cv_auc'],
        'search_results': search_results,
    }


In [ ]:
class PrefitScaledLogisticPipeline:
    def __init__(self, scaler, model):
        self.named_steps = {
            'standardscaler': scaler,
            'logisticregressioncv': model,
        }

    def predict_proba(self, x_matrix):
        return self.named_steps['logisticregressioncv'].predict_proba(
            self.named_steps['standardscaler'].transform(x_matrix)
        )

    def predict(self, x_matrix):
        return self.named_steps['logisticregressioncv'].predict(
            self.named_steps['standardscaler'].transform(x_matrix)
        )


def fit_elastic_net(x_matrix, y, alpha_grid=np.round(np.arange(0, 1.01, 0.1), 1), nfolds=5, seed=123, prefit_scaler=None):
    y_array = np.asarray(y, dtype=float)
    class_counts = pd.Series(y_array).value_counts()
    folds = int(min(nfolds, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two outcome classes with at least two rows each for elastic-net CV.')

    if not RUN_CPU_ONLY_COMPARISON_MODELS:
        raise RuntimeError(
            'fit_elastic_net uses sklearn LogisticRegressionCV, which trains on CPU. '
            'Set RUN_CPU_ONLY_COMPARISON_MODELS = True to run this CPU comparison model.'
        )

    results = []
    best_pipeline = None
    best_auc = -np.inf
    best_alpha = np.nan
    best_lambda = np.nan
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)

    for alpha in alpha_grid:
        penalty = 'l2' if alpha == 0 else 'elasticnet'
        l1_ratios = None if alpha == 0 else [float(alpha)]
        cv_model = LogisticRegressionCV(
            Cs=np.logspace(-4, 4, 30),
            cv=cv,
            penalty=penalty,
            solver='saga',
            l1_ratios=l1_ratios,
            scoring='roc_auc',
            max_iter=10000,
            random_state=seed,
            refit=True,
        )
        if prefit_scaler is None:
            pipeline = make_pipeline(StandardScaler(), cv_model)
            pipeline.fit(x_matrix, y_array)
            fitted = pipeline.named_steps['logisticregressioncv']
        else:
            x_scaled = prefit_scaler.transform(x_matrix)
            cv_model.fit(x_scaled, y_array)
            fitted = cv_model
            pipeline = PrefitScaledLogisticPipeline(prefit_scaler, fitted)
        scores = fitted.scores_[1.0]
        auc_cv = float(np.nanmax(np.nanmean(scores, axis=0)))
        lambda_value = float(1 / fitted.C_[0])
        results.append({'alpha': float(alpha), 'lambda': lambda_value, 'cv_auc': auc_cv})
        if auc_cv > best_auc:
            best_auc = auc_cv
            best_alpha = float(alpha)
            best_lambda = lambda_value
            best_pipeline = pipeline

    return {
        'best_model': best_pipeline,
        'best_alpha': best_alpha,
        'best_lambda': best_lambda,
        'best_auc': best_auc,
        'search_results': pd.DataFrame(results).sort_values('cv_auc', ascending=False).reset_index(drop=True),
    }


In [ ]:
def build_uplift_results(base_df, pred_treated, pred_control):
    results = base_df.copy()
    results['pred_ed_if_treated'] = pred_treated
    results['pred_ed_if_control'] = pred_control
    results['benefit_score'] = results['pred_ed_if_control'] - results['pred_ed_if_treated']
    results['uplift_bad_outcome'] = results['pred_ed_if_treated'] - results['pred_ed_if_control']
    results['uplift_decile'] = ntile_desc(results['benefit_score'], 10).to_numpy()
    return results


def summarize_uplift_deciles(results):
    return (
        results
        .groupby('uplift_decile', as_index=False)
        .agg(
            n=('outcome_ed_90d', 'size'),
            avg_benefit_score=('benefit_score', 'mean'),
            observed_ed_rate=('outcome_ed_90d', 'mean'),
            treated_pct=('intervention_flag', 'mean'),
            avg_pred_ed_if_treated=('pred_ed_if_treated', 'mean'),
            avg_pred_ed_if_control=('pred_ed_if_control', 'mean'),
        )
        .sort_values('uplift_decile')
    )


def print_highest_benefit(results, label, n=20):
    print(f'{label} top {n} highest-benefit members:')
    display(
        results
        .sort_values('benefit_score', ascending=False)
        [['outcome_ed_90d', 'intervention_flag', 'pred_ed_if_treated',
          'pred_ed_if_control', 'benefit_score', 'uplift_decile']]
        .head(n)
    )
    print()


def glmnet_contribution_importance_frame(model_info, x_matrix, label):
    pipeline = model_info['best_model']
    scaler = pipeline.named_steps['standardscaler']
    glmnet_model = pipeline.named_steps['logisticregressioncv']
    x_scaled = scaler.transform(x_matrix)
    coefficients = glmnet_model.coef_.ravel()
    contributions = x_scaled * coefficients
    return (
        pd.DataFrame({
            'feature': list(x_matrix.columns),
            'mean_abs_model_contribution': np.abs(contributions).mean(axis=0),
            'coefficient': coefficients,
            'model': label,
            'importance_type': 'standardized_logit_contribution',
        })
        .sort_values('mean_abs_model_contribution', ascending=False)
        .reset_index(drop=True)
    )


def save_bar_chart(df, x_col, y_col, title, x_label, y_label, path, y_max=None):
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.bar(df[x_col].astype(str), df[y_col], color='#4f76b5')
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    if y_max is not None:
        ax.set_ylim(0, y_max)
    ax.grid(axis='y', alpha=0.35)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)


---
## 4. Load & Preprocess GenRocket Data
---


In [ ]:
# --- Load GenRocket dataset ---
raw = pd.read_csv(PROJECT_ROOT / 'DataSets' / 'Genrocket_10k_seed1_updated.csv', low_memory=False)
print(f'Raw GenRocket dataset: {raw.shape}')

# --- Column renames to match original workflow ---
df = raw.rename(columns={
    'Clinical Score': 'percolator_clinical_score',
    'SDOH Score': 'percolator_sdoh_score',
    'utilization_score': 'percolator_utilization_score',
    'currentRiskScore': 'current_risk_score',
})

# --- Convert Y/N binary flags to 0/1 ---
binary_cols = [
    'living_alone_flag', 'dual_eligible', 'diabetes_flag', 'chf_flag', 'copd_flag',
    'asthma_flag', 'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag',
    'pregnancy_flag', 'behavioral_health_risk_flag', 'food_insecurity_flag',
    'housing_instability_flag', 'transportation_barrier_flag', 'utilities_insecurity_flag',
    'high_cost_drug_flag', 'opioid_flag', 'polypharmacy_flag',
    'notes_escalation_flag', 'community_referral_flag', 'pharmacy_review_flag',
    'intervention_flag',
]
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper().map({'Y': 1, 'N': 0, '1': 1, '0': 0}).fillna(0).astype(int)

# --- Convert 'Null' strings to NaN, then to numeric ---
numeric_cols = [
    'age', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d',
    'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m',
    'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc',
    'percolator_clinical_score', 'percolator_sdoh_score', 'percolator_utilization_score',
    'current_risk_score', 'intervention_days_active', 'touches_per_month',
    'outreach_attempts', 'successful_contacts', 'avg_call_duration_min',
    'max_call_duration_min', 'days_to_intervention_start',
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].replace({'Null': np.nan, 'NULL': np.nan, '': np.nan}), errors='coerce')

# Convert outcome
df['outcome_ed_90d'] = pd.to_numeric(df['outcome_ed_90d'], errors='coerce').fillna(0).astype(int)

print(f'Processed dataset: {df.shape}')
print(f'Treatment rate: {df["intervention_flag"].mean():.1%}')
print(f'Outcome rate: {df["outcome_ed_90d"].mean():.1%}')


---
## 4b. Dataset Investigation (Pre-Modeling Validation)
---

This section analyzes the full dataset BEFORE any train/test split, standardization, or encoding.
It checks whether variables relate realistically, flags potential leakage, and validates that
the synthetic data looks plausible for uplift modeling.


In [ ]:
investigation_folder = ensure_output_folder(PROJECT_ROOT / 'Outputs' / 'upliftGenrocket' / 'Data_Investigation')
print('Investigation outputs:', investigation_folder)


In [ ]:
# Dataset overview
n_rows, n_cols = df.shape
outcome_col = 'outcome_ed_90d'
treatment_col = 'intervention_flag'

overview = pd.DataFrame({
    'Metric': ['Total rows', 'Total columns', 'Duplicate rows', 'Duplicate member_ids',
               'Outcome (outcome_ed_90d) prevalence', 'Treatment (intervention_flag) prevalence',
               'Numeric columns', 'Non-numeric columns', 'Columns with any missing', 'Mean missingness rate'],
    'Value': [n_rows, n_cols, df.duplicated().sum(),
              df['member_id'].duplicated().sum() if 'member_id' in df.columns else 'N/A',
              f"{df[outcome_col].mean():.1%}", f"{df[treatment_col].mean():.1%}",
              df.select_dtypes(include=[np.number]).shape[1],
              df.select_dtypes(exclude=[np.number]).shape[1],
              int((df.isna().sum() > 0).sum()),
              f"{df.isna().mean().mean():.1%}"]
})
overview.to_csv(investigation_folder / 'dataset_overview.csv', index=False)
display(overview)


In [ ]:
# Missingness by column
missingness = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isna().sum().values,
    'missing_pct': (df.isna().mean() * 100).round(2).values,
    'dtype': df.dtypes.astype(str).values,
    'nunique': [df[c].nunique(dropna=True) for c in df.columns],
}).sort_values('missing_pct', ascending=False).reset_index(drop=True)
missingness.to_csv(investigation_folder / 'missingness_summary.csv', index=False)
print(f"Columns with >0% missing: {(missingness['missing_pct'] > 0).sum()}")
display(missingness[missingness['missing_pct'] > 0])


### Numeric Feature Correlations


In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
# Exclude id/member_id
exclude_cols = ['id', 'member_id']
numeric_features = [c for c in numeric_df.columns if c not in exclude_cols]
num_df = numeric_df[numeric_features]

# Pearson
pearson_corr = num_df.corr(method='pearson')
pearson_corr.to_csv(investigation_folder / 'numeric_correlation_pearson.csv')

# Spearman
spearman_corr = num_df.corr(method='spearman')
spearman_corr.to_csv(investigation_folder / 'numeric_correlation_spearman.csv')

# Heatmap
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(pearson_corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax,
            xticklabels=True, yticklabels=True, linewidths=0.3)
ax.set_title('Numeric Feature Correlation Matrix (Pearson)')
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7)
fig.tight_layout()
fig.savefig(investigation_folder / 'numeric_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Correlation matrices saved ({len(numeric_features)} numeric features).')


In [ ]:
# Extract highly correlated pairs (|r| > 0.5, excluding self)
pairs = []
for i in range(len(pearson_corr.columns)):
    for j in range(i+1, len(pearson_corr.columns)):
        r = pearson_corr.iloc[i, j]
        if abs(r) > 0.5:
            pairs.append({
                'feature_1': pearson_corr.columns[i],
                'feature_2': pearson_corr.columns[j],
                'pearson_r': round(r, 4),
                'abs_pearson_r': round(abs(r), 4),
            })
high_corr_pairs = pd.DataFrame(pairs).sort_values('abs_pearson_r', ascending=False).reset_index(drop=True)
high_corr_pairs.to_csv(investigation_folder / 'high_correlation_pairs.csv', index=False)
print(f'Highly correlated pairs (|r| > 0.5): {len(high_corr_pairs)}')
display(high_corr_pairs.head(20))


### Predictor → Outcome Relationships


In [ ]:
# Numeric features vs outcome
outcome_corrs = num_df.corrwith(df[outcome_col], method='spearman').sort_values(key=abs, ascending=False)
outcome_relationship = pd.DataFrame({
    'feature': outcome_corrs.index,
    'spearman_vs_outcome': outcome_corrs.values.round(4),
    'abs_spearman': np.abs(outcome_corrs.values).round(4),
}).reset_index(drop=True)
outcome_relationship.to_csv(investigation_folder / 'outcome_relationship_summary.csv', index=False)
print('Top predictors correlated with outcome (Spearman):')
display(outcome_relationship.head(15))

# Bar chart
top_outcome = outcome_relationship.head(15).sort_values('spearman_vs_outcome')
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = ['#e45756' if v < 0 else '#4c78a8' for v in top_outcome['spearman_vs_outcome']]
ax.barh(top_outcome['feature'], top_outcome['spearman_vs_outcome'], color=colors)
ax.set_title('Top 15 Predictors vs outcome_ed_90d (Spearman)')
ax.set_xlabel('Spearman Correlation')
ax.axvline(0, color='gray', linewidth=0.8)
fig.tight_layout()
fig.savefig(investigation_folder / 'outcome_correlation_top15.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Binary flags vs outcome rate
binary_flag_cols = [c for c in df.columns if c.endswith('_flag') and c != outcome_col and c != treatment_col]
binary_outcome_rows = []
for col in binary_flag_cols:
    if col in df.columns and df[col].nunique() <= 2:
        rate_1 = df.loc[df[col] == 1, outcome_col].mean()
        rate_0 = df.loc[df[col] == 0, outcome_col].mean()
        n_1 = int((df[col] == 1).sum())
        binary_outcome_rows.append({
            'flag': col, 'n_flagged': n_1, 'flag_rate': n_1 / len(df),
            'outcome_rate_if_1': rate_1, 'outcome_rate_if_0': rate_0,
            'outcome_rate_difference': rate_1 - rate_0,
            'relative_risk': rate_1 / rate_0 if rate_0 > 0 else np.inf,
        })
binary_outcome_df = pd.DataFrame(binary_outcome_rows).sort_values('outcome_rate_difference', key=abs, ascending=False).reset_index(drop=True)
binary_outcome_df.to_csv(investigation_folder / 'binary_flags_vs_outcome.csv', index=False)
print('Binary flags vs outcome (sorted by outcome rate difference):')
display(binary_outcome_df)


In [ ]:
# Numeric features vs treatment
treatment_corrs = num_df.corrwith(df[treatment_col], method='spearman').sort_values(key=abs, ascending=False)
treatment_relationship = pd.DataFrame({
    'feature': treatment_corrs.index,
    'spearman_vs_treatment': treatment_corrs.values.round(4),
    'abs_spearman': np.abs(treatment_corrs.values).round(4),
}).reset_index(drop=True)
treatment_relationship.to_csv(investigation_folder / 'treatment_relationship_summary.csv', index=False)
print('Top predictors correlated with treatment (Spearman):')
display(treatment_relationship.head(15))

# Binary flags vs treatment rate
binary_treatment_rows = []
for col in binary_flag_cols:
    if col in df.columns and df[col].nunique() <= 2:
        rate_1 = df.loc[df[col] == 1, treatment_col].mean()
        rate_0 = df.loc[df[col] == 0, treatment_col].mean()
        n_1 = int((df[col] == 1).sum())
        binary_treatment_rows.append({
            'flag': col, 'n_flagged': n_1,
            'treatment_rate_if_1': rate_1, 'treatment_rate_if_0': rate_0,
            'treatment_rate_difference': rate_1 - rate_0,
        })
binary_treatment_df = pd.DataFrame(binary_treatment_rows).sort_values('treatment_rate_difference', key=abs, ascending=False).reset_index(drop=True)
binary_treatment_df.to_csv(investigation_folder / 'binary_flags_vs_treatment.csv', index=False)
print('Binary flags vs treatment assignment:')
display(binary_treatment_df)


### Leakage & Realism Checks

Flag any predictors with suspiciously strong outcome relationships that could indicate:
- Post-outcome information leaking into predictors
- Near-perfect separation (feature almost perfectly predicts the outcome)
- Unrealistic effect sizes for a synthetic dataset


In [ ]:
# Flag potential leakage: outcome correlations > 0.3 or outcome rate ratios > 5x
leakage_flags = []

# From numeric correlations
for _, row in outcome_relationship.iterrows():
    if abs(row['spearman_vs_outcome']) > 0.30:
        leakage_flags.append({
            'feature': row['feature'],
            'check_type': 'High numeric-outcome correlation',
            'metric': f"Spearman = {row['spearman_vs_outcome']:.4f}",
            'severity': 'HIGH' if abs(row['spearman_vs_outcome']) > 0.5 else 'MODERATE',
        })

# From binary flags
for _, row in binary_outcome_df.iterrows():
    if row['relative_risk'] > 5 or row['relative_risk'] < 0.2:
        leakage_flags.append({
            'feature': row['flag'],
            'check_type': 'Extreme outcome rate ratio',
            'metric': f"RR = {row['relative_risk']:.2f} (rate_1={row['outcome_rate_if_1']:.3f}, rate_0={row['outcome_rate_if_0']:.3f})",
            'severity': 'HIGH' if row['relative_risk'] > 10 else 'MODERATE',
        })

# From near-perfect correlation pairs
for _, row in high_corr_pairs.iterrows():
    if row['abs_pearson_r'] > 0.9:
        leakage_flags.append({
            'feature': f"{row['feature_1']} \u2194 {row['feature_2']}",
            'check_type': 'Near-perfect predictor correlation',
            'metric': f"Pearson r = {row['pearson_r']:.4f}",
            'severity': 'HIGH',
        })

leakage_df = pd.DataFrame(leakage_flags).sort_values('severity', ascending=True).reset_index(drop=True)
leakage_df.to_csv(investigation_folder / 'potential_leakage_flags.csv', index=False)
print(f'Potential leakage flags: {len(leakage_df)} ({(leakage_df["severity"] == "HIGH").sum()} HIGH)')
display(leakage_df)


### Key Variable Distributions


In [ ]:
# Outcome rate by risk_tier
if 'risk_tier' in df.columns:
    risk_outcome = df.groupby('risk_tier', observed=False).agg(
        n=(outcome_col, 'size'),
        outcome_rate=(outcome_col, 'mean'),
        treatment_rate=(treatment_col, 'mean'),
    ).reset_index()
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(risk_outcome['risk_tier'].astype(str), risk_outcome['outcome_rate'], color='#4c78a8')
    ax.set_title('Outcome Rate by Risk Tier')
    ax.set_xlabel('Risk Tier')
    ax.set_ylabel('Outcome Rate (ED 90d)')
    ax.grid(axis='y', alpha=0.3)
    for i, (_, r) in enumerate(risk_outcome.iterrows()):
        ax.text(i, r['outcome_rate'] + 0.005, f"{r['outcome_rate']:.1%}", ha='center', fontsize=9)
    fig.tight_layout()
    fig.savefig(investigation_folder / 'outcome_rate_by_risk_tier.png', dpi=150)
    plt.show()
    display(risk_outcome)

# Histograms for key numeric variables
key_numerics = ['age', 'ed_visits_last_6m', 'admits_last_6m', 'total_cost_last_6m',
                'current_risk_score', 'percolator_clinical_score', 'med_adherence_pdc']
key_numerics = [c for c in key_numerics if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(key_numerics):
    vals = pd.to_numeric(df[col], errors='coerce').dropna()
    if vals.empty:
        continue
    axes[i].hist(vals, bins=30, edgecolor='white', color='#4c78a8')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('Count')
for j in range(len(key_numerics), len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Key Numeric Variable Distributions', fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(investigation_folder / 'key_numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Outcome rate by key categorical variables
cat_cols_to_check = ['risk_tier', 'program', 'client_contract', 'service_region', 'gender', 'plan_type']
cat_cols_to_check = [c for c in cat_cols_to_check if c in df.columns]

for col in cat_cols_to_check:
    cat_outcome = df.groupby(col, observed=False).agg(
        n=(outcome_col, 'size'), outcome_rate=(outcome_col, 'mean'),
    ).sort_values('outcome_rate', ascending=False).reset_index()
    if len(cat_outcome) > 15:
        cat_outcome = cat_outcome.head(15)
    fig, ax = plt.subplots(figsize=(8, max(3.5, 0.35 * len(cat_outcome))))
    ax.barh(cat_outcome[col].astype(str), cat_outcome['outcome_rate'], color='#59a14f')
    ax.set_title(f'Outcome Rate by {col}')
    ax.set_xlabel('ED 90d Rate')
    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()
    fig.savefig(investigation_folder / f'outcome_rate_by_{col}.png', dpi=150, bbox_inches='tight')
    plt.show()


### High-Risk Relationship Summary

Ranked tables of strongest relationships — useful for quick realism assessment.


In [ ]:
# Strongest relationships summary
print('=== STRONGEST POSITIVE RELATIONSHIPS WITH OUTCOME ===')
display(outcome_relationship[outcome_relationship['spearman_vs_outcome'] > 0].head(10))

print('\n=== STRONGEST NEGATIVE RELATIONSHIPS WITH OUTCOME ===')
display(outcome_relationship[outcome_relationship['spearman_vs_outcome'] < 0].head(10))

print('\n=== STRONGEST RELATIONSHIPS WITH TREATMENT ===')
display(treatment_relationship.head(10))

print('\n=== MOST CORRELATED PREDICTOR PAIRS ===')
display(high_corr_pairs.head(10))


### Investigation Summary


In [ ]:
print('='*60)
print('DATASET INVESTIGATION SUMMARY')
print('='*60)
print(f"Total rows: {n_rows:,}")
print(f"Total columns: {n_cols}")
print(f"Outcome prevalence: {df[outcome_col].mean():.1%}")
print(f"Treatment prevalence: {df[treatment_col].mean():.1%}")
print(f"Highly correlated pairs (|r|>0.5): {len(high_corr_pairs)}")
print(f"Potential leakage flags: {len(leakage_df)} ({(leakage_df['severity'] == 'HIGH').sum()} HIGH)")
print(f"\nAll investigation outputs saved to: {investigation_folder}")
print('='*60)


---
## 5. Select Predictors & Build Modeling Frame
---


In [ ]:
candidate_predictors_all = [
    'client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender',
    'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag', 'diabetes_flag',
    'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag',
    'substance_use_flag', 'ckd_flag', 'pregnancy_flag', 'behavioral_health_risk_flag',
    'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag',
    'utilities_insecurity_flag', 'pcp_visits_last_6m', 'specialist_visits_last_6m',
    'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m',
    'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag',
    'opioid_flag', 'polypharmacy_flag', 'percolator_utilization_score',
    'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'risk_tier',
    'intervention_type', 'intervention_days_active', 'touches_per_month', 'outreach_attempts',
    'successful_contacts', 'avg_call_duration_min', 'max_call_duration_min', 'notes_escalation_flag',
    'community_referral_flag', 'pharmacy_review_flag', 'engagement_level',
    'days_to_intervention_start', 'intervention_start_month', 'intervention_start_wday',
]

candidate_predictors = [col for col in candidate_predictors_all if col in df.columns]
missing_predictors = [col for col in candidate_predictors_all if col not in df.columns]

if missing_predictors:
    print('Predictors not found in dataset:', missing_predictors)
else:
    print('All candidate predictors are present in dataset.')

model_df = df[['outcome_ed_90d', 'intervention_flag', *candidate_predictors]].copy()
model_df = model_df[model_df['outcome_ed_90d'].notna() & model_df['intervention_flag'].notna()].copy()
print(f'Modeling rows after dropping missing outcome/treatment: {len(model_df)}')


---
## 6. Data Type Handling & Imputation
---


In [ ]:
possible_numeric_cols = [
    'age', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d',
    'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m',
    'rx_count_last_6m', 'med_adherence_pdc', 'percolator_utilization_score',
    'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score',
    'intervention_days_active', 'touches_per_month', 'outreach_attempts', 'successful_contacts',
    'avg_call_duration_min', 'max_call_duration_min', 'days_to_intervention_start',
    'intervention_start_month', 'intervention_start_wday',
]

flag_like_cols = [col for col in model_df.columns if col.endswith('_flag')]
for col in flag_like_cols:
    model_df[col] = to_binary(model_df[col])

for col in present_columns(possible_numeric_cols, model_df):
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

for col in model_df.columns:
    if col in ['outcome_ed_90d', 'intervention_flag']:
        continue
    if not pd.api.types.is_numeric_dtype(model_df[col]):
        model_df[col] = impute_categorical(model_df[col])

for col in model_df.columns:
    if col in ['outcome_ed_90d', 'intervention_flag']:
        continue
    if pd.api.types.is_numeric_dtype(model_df[col]):
        model_df[col] = impute_numeric(model_df[col])

unique_counts = model_df.apply(lambda c: c.dropna().nunique())
keep_cols = list(unique_counts[unique_counts > 1].index)
model_df = model_df.loc[:, keep_cols].copy().reset_index(drop=True)
model_df.insert(0, 'member_id', np.arange(len(model_df), dtype=int))

print('Final modeling columns:', len(model_df.columns))
print(list(model_df.columns))


---
## 7. Train / Test Split
---


In [ ]:
train_df, test_df = split_train_test(
    model_df,
    train_fraction=0.70,
    seed=123,
    stratify_columns=['intervention_flag', 'outcome_ed_90d'],
)

print('Training rows:', len(train_df))
print('Testing rows:', len(test_df))


---
## 8. Separate Treated / Control
---


In [ ]:
train_treated = train_df[train_df['intervention_flag'] == 1].copy()
train_control = train_df[train_df['intervention_flag'] == 0].copy()

print('Training treated rows:', len(train_treated))
print('Training control rows:', len(train_control))

if len(train_treated) < 50:
    raise ValueError('Too few treated rows to train a stable model.')
if len(train_control) < 50:
    raise ValueError('Too few control rows to train a stable model.')


---
## 9. Build Model Matrices
---


In [ ]:
id_cols = ['member_id']
feature_cols = [
    col for col in model_df.columns
    if col not in ['outcome_ed_90d', 'intervention_flag', *id_cols]
]

train_treated_x_df = train_treated[feature_cols].copy()
train_control_x_df = train_control[feature_cols].copy()
test_x_df = test_df[feature_cols].copy()

combined_matrix, split_matrices = make_design_matrix([train_treated_x_df, train_control_x_df, test_x_df])
x_treated, x_control, x_test = split_matrices

y_treated = train_treated['outcome_ed_90d'].astype(float).to_numpy()
y_control = train_control['outcome_ed_90d'].astype(float).to_numpy()

print(f'Model matrix dimensions: {x_test.shape[1]} columns')
print(f'Treated training: {x_treated.shape}, Control training: {x_control.shape}, Test: {x_test.shape}')


---
## 10. Data Review Summary
---


In [ ]:
def is_binary_indicator_column(series):
    non_missing = pd.Series(series).dropna()
    if non_missing.empty:
        return False
    numeric_values = pd.to_numeric(non_missing, errors='coerce')
    if numeric_values.isna().any():
        return False
    return set(numeric_values.unique()).issubset({0, 1}) and numeric_values.nunique() <= 2


model_feature_type_rows = []
for col in feature_cols:
    series = model_df[col]
    if is_binary_indicator_column(series):
        predictor_type = 'Binary indicator'
    elif col in possible_numeric_cols or pd.api.types.is_numeric_dtype(series):
        predictor_type = 'Continuous/count numeric'
    else:
        predictor_type = 'Multi-level categorical'
    model_feature_type_rows.append({'variable': col, 'predictor_type': predictor_type})

model_feature_type_summary = pd.DataFrame(model_feature_type_rows)
predictor_type_counts = model_feature_type_summary['predictor_type'].value_counts().to_dict()

continuous_count_predictors = int(predictor_type_counts.get('Continuous/count numeric', 0))
binary_indicator_predictors = int(predictor_type_counts.get('Binary indicator', 0))
multilevel_categorical_predictors = int(predictor_type_counts.get('Multi-level categorical', 0))

data_review_summary = pd.DataFrame([{
    'total_members': len(model_df),
    'treated_members': int((model_df['intervention_flag'] == 1).sum()),
    'control_members': int((model_df['intervention_flag'] == 0).sum()),
    'treatment_rate': float(model_df['intervention_flag'].mean()),
    'outcome_events': int((model_df['outcome_ed_90d'] == 1).sum()),
    'outcome_prevalence': float(model_df['outcome_ed_90d'].mean()),
    'treated_outcome_rate': float(model_df.loc[model_df['intervention_flag'] == 1, 'outcome_ed_90d'].mean()),
    'control_outcome_rate': float(model_df.loc[model_df['intervention_flag'] == 0, 'outcome_ed_90d'].mean()),
    'number_of_predictors': len(feature_cols),
    'number_of_continuous_count_predictors': continuous_count_predictors,
    'number_of_binary_indicator_predictors': binary_indicator_predictors,
    'number_of_multilevel_categorical_predictors': multilevel_categorical_predictors,
    'number_of_model_matrix_columns': x_test.shape[1],
}])

data_review_summary.to_csv(output_folder / 'data_review_summary.csv', index=False)
print('Data review summary:')
display(data_review_summary)


---
## 11. Train XGBoost T-Learner (Baseline)
---


In [ ]:
print('Unique y_treated values:', np.sort(pd.unique(y_treated)))
print('Unique y_control values:', np.sort(pd.unique(y_control)))

dtrain_treated = make_dmatrix(x_treated, y_treated)
dtrain_control = make_dmatrix(x_control, y_control)

params = xgb_training_params(
    XGB_GPU_PARAMS,
    {'max_depth': 4, 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8},
    eval_metric='logloss',
    seed=123,
)

model_treated = xgb.train(params=params, dtrain=dtrain_treated, num_boost_round=150, verbose_eval=False)
model_control = xgb.train(params=params, dtrain=dtrain_control, num_boost_round=150, verbose_eval=False)
assert_xgb_booster_uses_cuda(model_treated, 'Baseline treated XGBoost model')
assert_xgb_booster_uses_cuda(model_control, 'Baseline control XGBoost model')

test_treated_pos = np.where(test_df['intervention_flag'].to_numpy() == 1)[0]
test_control_pos = np.where(test_df['intervention_flag'].to_numpy() == 0)[0]

pred_treated = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated, 'XGBoost Treated model')

pred_control = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control, 'XGBoost Control model')


---
## 12. XGBoost CV Grid Search — Treated & Control Models
---


In [ ]:
xgb_grid = [
    {'max_depth': md, 'eta': eta, 'min_child_weight': mcw}
    for md, eta, mcw in product([3, 4, 5], [0.03, 0.05, 0.10], [1, 5])
]

xgb_treated_cv = fit_xgb_cv_grid(x_matrix=x_treated, y=y_treated, grid=xgb_grid, nrounds_max=500, nfold=5)
model_treated = xgb_treated_cv['model']
assert_xgb_booster_uses_cuda(model_treated, 'CV-tuned treated XGBoost model')

print('XGBoost Treated best CV AUC:', round(xgb_treated_cv['best_cv_auc'], 4))
print('XGBoost Treated best nrounds:', xgb_treated_cv['best_nrounds'])
print('XGBoost Treated best params:', xgb_treated_cv['best_params'])
print()

xgb_control_cv = fit_xgb_cv_grid(x_matrix=x_control, y=y_control, grid=xgb_grid, nrounds_max=500, nfold=5)
model_control = xgb_control_cv['model']
assert_xgb_booster_uses_cuda(model_control, 'CV-tuned control XGBoost model')

print('XGBoost Control best CV AUC:', round(xgb_control_cv['best_cv_auc'], 4))
print('XGBoost Control best nrounds:', xgb_control_cv['best_nrounds'])
print('XGBoost Control best params:', xgb_control_cv['best_params'])


In [ ]:
pred_treated_cv_xgb = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated_cv_xgb, 'XGBoost Treated CV-tuned test')

pred_control_cv_xgb = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control_cv_xgb, 'XGBoost Control CV-tuned test')

print('CV-tuned XGBoost models trained successfully.')


---
## 13. GLMNet T-Learner (CPU Comparison)
---


In [ ]:
enet_treated = None
enet_control = None

if RUN_CPU_ONLY_COMPARISON_MODELS:
    print('Training GLMNet comparison models on CPU with sklearn LogisticRegressionCV.')
    glmnet_shared_scaler = StandardScaler().fit(pd.concat([x_treated, x_control], axis=0))
    print('GLMNet shared scaler fit on combined treated/control training matrix.')

    enet_treated = fit_elastic_net(x_treated, y_treated, prefit_scaler=glmnet_shared_scaler)
    print('Best treated alpha:', enet_treated['best_alpha'])
    print('Best treated lambda:', enet_treated['best_lambda'])
    print('Best treated CV AUC:', round(enet_treated['best_auc'], 4))
    print()

    enet_control = fit_elastic_net(x_control, y_control, prefit_scaler=glmnet_shared_scaler)
    print('Best control alpha:', enet_control['best_alpha'])
    print('Best control lambda:', enet_control['best_lambda'])
    print('Best control CV AUC:', round(enet_control['best_auc'], 4))
else:
    print('Skipping GLMNet models (RUN_CPU_ONLY_COMPARISON_MODELS = False).')


---
## 14. Score Test Set & Build Uplift Results
---


In [ ]:
# XGBoost scoring
p_treated_xgboost = model_treated.predict(make_dmatrix(x_test))
p_control_xgboost = model_control.predict(make_dmatrix(x_test))

results_test_xgboost = build_uplift_results(test_df, p_treated_xgboost, p_control_xgboost)
decile_summary_xgboost = summarize_uplift_deciles(results_test_xgboost)

print('XGBoost T-Learner decile summary:')
display(decile_summary_xgboost)
print_highest_benefit(results_test_xgboost, 'XGBoost T-Learner')

results_test_xgboost.to_csv(xgboost_output_path, index=False)
decile_summary_xgboost.to_csv(xgboost_summary_path, index=False)
print('XGBoost scored output saved to:', xgboost_output_path)

# GLMNet scoring
results_test_glmnet = None
decile_summary_glmnet = None
p_treated_glmnet = None
p_control_glmnet = None

if enet_treated is not None and enet_control is not None:
    p_treated_glmnet = enet_treated['best_model'].predict_proba(x_test)[:, 1]
    p_control_glmnet = enet_control['best_model'].predict_proba(x_test)[:, 1]

    results_test_glmnet = build_uplift_results(test_df, p_treated_glmnet, p_control_glmnet)
    decile_summary_glmnet = summarize_uplift_deciles(results_test_glmnet)

    print('\nGLMNet T-Learner decile summary:')
    display(decile_summary_glmnet)
    print_highest_benefit(results_test_glmnet, 'GLMNet T-Learner')

    results_test_glmnet.to_csv(glmnet_output_path, index=False)
    decile_summary_glmnet.to_csv(glmnet_summary_path, index=False)
    print('GLMNet scored output saved to:', glmnet_output_path)
else:
    print('Skipped GLMNet scoring.')


---
## 15. SHAP / Risk Driver Importance
---


In [ ]:
def save_model_driver_outputs(treated_importance, control_importance, folder, model_label, value_col, x_label, csv_name='shap_importance_treated_control_models.csv'):
    combined = pd.concat([treated_importance, control_importance], ignore_index=True)
    combined.to_csv(folder / csv_name, index=False)
    for driver_df, treatment_label, filename in [
        (treated_importance, 'Treated Model', 'dashboard_shap_treated_model.png'),
        (control_importance, 'Control Model', 'dashboard_shap_control_model.png'),
    ]:
        top = driver_df.nlargest(20, value_col).sort_values(value_col)
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(top['feature'], top[value_col])
        ax.set_title(f'{model_label}: Top Risk Drivers - {treatment_label}')
        ax.set_xlabel(x_label)
        ax.set_ylabel('Feature')
        fig.tight_layout()
        fig.savefig(folder / filename, dpi=150)
        plt.close(fig)
    print(f'{model_label} risk-driver outputs saved to:', folder)
    return combined


shap_treated_importance = shap_importance_frame(model_treated, x_test, 'Treated Model')
shap_control_importance = shap_importance_frame(model_control, x_test, 'Control Model')
shap_importance_combined = save_model_driver_outputs(
    shap_treated_importance, shap_control_importance,
    xgboost_output_folder, 'XGBoost', 'mean_abs_shap', 'Mean Absolute SHAP Contribution',
)

if enet_treated is not None and enet_control is not None:
    glmnet_driver_treated = glmnet_contribution_importance_frame(enet_treated, x_test, 'Treated Model')
    glmnet_driver_control = glmnet_contribution_importance_frame(enet_control, x_test, 'Control Model')
    glmnet_driver_combined = save_model_driver_outputs(
        glmnet_driver_treated, glmnet_driver_control,
        glmnet_output_folder, 'GLMNet', 'mean_abs_model_contribution',
        'Mean Absolute Standardized Logit Contribution',
    )


---
## 16. Benefit Score Driver Importance
---


In [ ]:
def save_benefit_driver_outputs(importance_df, folder, model_label, value_col, x_label):
    importance_df.to_csv(folder / 'shap_importance_benefit_score.csv', index=False)
    top = importance_df.head(20).sort_values(value_col)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top[value_col])
    ax.set_title(f'{model_label}: Top Drivers of Predicted Treatment Benefit')
    ax.set_xlabel(x_label)
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_shap_benefit_score.png', dpi=150)
    plt.close(fig)
    print(f'{model_label} benefit-driver outputs saved to:', folder)


def xgboost_benefit_shap_importance_frame(model_t, model_c, x_matrix):
    benefit_dtest = make_dmatrix(x_matrix)
    control_contribs = model_c.predict(benefit_dtest, pred_contribs=True)
    treated_contribs = model_t.predict(benefit_dtest, pred_contribs=True)
    shap_columns = [*x_matrix.columns, 'BIAS']
    control_shap = pd.DataFrame(control_contribs, columns=shap_columns, index=x_matrix.index)
    treated_shap = pd.DataFrame(treated_contribs, columns=shap_columns, index=x_matrix.index)
    benefit_shap = control_shap - treated_shap
    benefit_shap_no_bias = benefit_shap.drop(columns=['BIAS'], errors='ignore')
    return (
        pd.DataFrame({
            'feature': benefit_shap_no_bias.columns,
            'mean_abs_benefit_shap': benefit_shap_no_bias.abs().mean(axis=0).to_numpy(),
            'mean_signed_benefit_shap': benefit_shap_no_bias.mean(axis=0).to_numpy(),
            'pct_positive_benefit_shap': benefit_shap_no_bias.gt(0).mean(axis=0).to_numpy(),
            'importance_type': 'xgboost_raw_margin_shap_difference',
        })
        .sort_values('mean_abs_benefit_shap', ascending=False)
        .reset_index(drop=True)
    )


xgboost_benefit_shap_importance = xgboost_benefit_shap_importance_frame(model_treated, model_control, x_test)
print('Top XGBoost SHAP features driving predicted treatment benefit:')
display(xgboost_benefit_shap_importance.head(20))
save_benefit_driver_outputs(xgboost_benefit_shap_importance, xgboost_output_folder, 'XGBoost', 'mean_abs_benefit_shap', 'Mean Absolute SHAP Contribution to Benefit')

if enet_treated is not None and enet_control is not None:
    def glmnet_benefit_contribution_importance_frame(treated_info, control_info, x_matrix):
        t_pipeline = treated_info['best_model']
        c_pipeline = control_info['best_model']
        t_scaler = t_pipeline.named_steps['standardscaler']
        c_scaler = c_pipeline.named_steps['standardscaler']
        t_model = t_pipeline.named_steps['logisticregressioncv']
        c_model = c_pipeline.named_steps['logisticregressioncv']
        t_contrib = pd.DataFrame(t_scaler.transform(x_matrix) * t_model.coef_.ravel(), columns=x_matrix.columns, index=x_matrix.index)
        c_contrib = pd.DataFrame(c_scaler.transform(x_matrix) * c_model.coef_.ravel(), columns=x_matrix.columns, index=x_matrix.index)
        benefit_contrib = c_contrib - t_contrib
        return (
            pd.DataFrame({
                'feature': benefit_contrib.columns,
                'mean_abs_benefit_contribution': benefit_contrib.abs().mean(axis=0).to_numpy(),
                'mean_signed_benefit_contribution': benefit_contrib.mean(axis=0).to_numpy(),
                'pct_positive_benefit_contribution': benefit_contrib.gt(0).mean(axis=0).to_numpy(),
                'importance_type': 'glmnet_standardized_logit_contribution_difference',
            })
            .sort_values('mean_abs_benefit_contribution', ascending=False)
            .reset_index(drop=True)
        )

    glmnet_benefit_importance = glmnet_benefit_contribution_importance_frame(enet_treated, enet_control, x_test)
    print('Top GLMNet features driving predicted treatment benefit:')
    display(glmnet_benefit_importance.head(20))
    save_benefit_driver_outputs(glmnet_benefit_importance, glmnet_output_folder, 'GLMNet', 'mean_abs_benefit_contribution', 'Mean Absolute Standardized Logit Contribution to Benefit')
else:
    glmnet_benefit_importance = None


---
## 17. Probability Calibration & Brier Scores
---


In [ ]:
def brier_scores_for_results(results, model_label):
    if results is None:
        return None
    rows = []
    for group_label, tv, pred_col in [('Treated', 1, 'pred_ed_if_treated'), ('Control', 0, 'pred_ed_if_control')]:
        sub = results[results['intervention_flag'] == tv].copy()
        if sub.empty:
            rows.append({'model': model_label, 'group': group_label, 'n': 0, 'brier_score': np.nan})
            continue
        rows.append({
            'model': model_label, 'group': group_label, 'n': len(sub),
            'observed_ed_rate': sub['outcome_ed_90d'].mean(),
            'avg_predicted_ed_rate': sub[pred_col].mean(),
            'brier_score': brier_score_loss(sub['outcome_ed_90d'], sub[pred_col]),
        })
    return pd.DataFrame(rows)


def save_calibration_plot(calibration_by_decile, folder, model_label):
    if calibration_by_decile is None or calibration_by_decile.empty:
        return
    groups = [g for g in ['Control', 'Treated'] if g in set(calibration_by_decile['group'])]
    fig, axes = plt.subplots(1, len(groups), figsize=(7 * len(groups), 5), sharey=True)
    if len(groups) == 1:
        axes = [axes]
    max_rate = max(calibration_by_decile['avg_predicted_ed_rate'].max(), calibration_by_decile['observed_ed_rate'].max(), 0.01)
    for ax, group_label in zip(axes, groups):
        gdf = calibration_by_decile[calibration_by_decile['group'] == group_label].sort_values('pred_risk_decile')
        x_vals = np.arange(len(gdf))
        bw = 0.38
        ax.bar(x_vals - bw/2, gdf['avg_predicted_ed_rate'], width=bw, label='Predicted', color='#4C78A8')
        ax.bar(x_vals + bw/2, gdf['observed_ed_rate'], width=bw, label='Observed', color='#F58518')
        ax.set_title(f'{model_label}: {group_label} Calibration')
        ax.set_xlabel('Predicted Risk Decile')
        ax.set_xticks(x_vals)
        ax.set_xticklabels(gdf['pred_risk_decile'].astype(str))
        ax.set_ylim(0, max_rate * 1.22)
        ax.grid(axis='y', alpha=0.25)
    axes[0].set_ylabel('ED Rate')
    axes[-1].legend(loc='upper right')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_calibration_plot.png', dpi=150, bbox_inches='tight')
    plt.close(fig)


def calibration_tables_for_results(results, model_label):
    if results is None:
        return None, None
    parts = []
    for group_label, tv, pred_col in [('Treated', 1, 'pred_ed_if_treated'), ('Control', 0, 'pred_ed_if_control')]:
        sub = results[results['intervention_flag'] == tv].copy()
        if sub.empty:
            continue
        sub['pred_risk_decile'] = ntile_desc(sub[pred_col], 10).to_numpy()
        by_d = sub.groupby('pred_risk_decile', as_index=False).agg(
            n=('outcome_ed_90d', 'size'),
            avg_predicted_ed_rate=(pred_col, 'mean'),
            observed_ed_rate=('outcome_ed_90d', 'mean'),
        ).sort_values('pred_risk_decile')
        by_d.insert(0, 'group', group_label)
        by_d.insert(0, 'model', model_label)
        by_d['calibration_error'] = by_d['observed_ed_rate'] - by_d['avg_predicted_ed_rate']
        by_d['abs_calibration_error'] = by_d['calibration_error'].abs()
        parts.append(by_d)
    if not parts:
        return None, None
    calibration_by_decile = pd.concat(parts, ignore_index=True)
    cal_summary_rows = []
    for (mn, gl), fr in calibration_by_decile.groupby(['model', 'group']):
        cal_summary_rows.append({'model': mn, 'group': gl, 'n': fr['n'].sum(),
            'mean_abs_calibration_error': fr['abs_calibration_error'].mean(),
            'max_abs_calibration_error': fr['abs_calibration_error'].max()})
    return calibration_by_decile, pd.DataFrame(cal_summary_rows)


def save_probability_evaluation_outputs(results, folder, model_label):
    brier_df = brier_scores_for_results(results, model_label)
    cal_by_d, cal_summary = calibration_tables_for_results(results, model_label)
    if brier_df is not None:
        brier_df.to_csv(folder / 'model_brier_scores.csv', index=False)
        print(f'{model_label} Brier scores:')
        display(brier_df)
    if cal_by_d is not None:
        cal_by_d.to_csv(folder / 'calibration_by_decile.csv', index=False)
        cal_summary.to_csv(folder / 'calibration_summary.csv', index=False)
        save_calibration_plot(cal_by_d, folder, model_label)
        print(f'{model_label} calibration summary:')
        display(cal_summary)
    return brier_df, cal_by_d, cal_summary


save_probability_evaluation_outputs(results_test_xgboost, xgboost_output_folder, 'XGBoost')
if results_test_glmnet is not None:
    save_probability_evaluation_outputs(results_test_glmnet, glmnet_output_folder, 'GLMNet')


---
## 18. Dashboard Charts & Uplift Curves
---


In [ ]:
def save_decile_dashboard_charts(decile_df, folder, model_label):
    # Benefit by decile
    save_bar_chart(decile_df, 'uplift_decile', 'avg_benefit_score',
        f'{model_label}: Average Predicted Benefit by Uplift Decile',
        'Uplift Decile: 1 = Highest Predicted Benefit', 'Average Benefit Score',
        folder / 'dashboard_avg_benefit_by_decile.png')
    # Predicted treated vs control
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    x_vals = np.arange(len(decile_df))
    bw = 0.35
    ax.bar(x_vals - bw/2, decile_df['avg_pred_ed_if_treated'], bw, label='Treated', color='#e45756')
    ax.bar(x_vals + bw/2, decile_df['avg_pred_ed_if_control'], bw, label='Control', color='#4c78a8')
    ax.set_xticks(x_vals)
    ax.set_xticklabels(decile_df['uplift_decile'].astype(str))
    ax.set_title(f'{model_label}: Predicted ED Rate by Treatment Group & Decile')
    ax.set_xlabel('Uplift Decile')
    ax.set_ylabel('Predicted ED Rate')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_predicted_treated_vs_control.png', dpi=150)
    plt.close(fig)


def save_uplift_curve(results, folder, model_label):
    sorted_r = results.sort_values('benefit_score', ascending=False).reset_index(drop=True)
    sorted_r['cumulative_benefit'] = sorted_r['benefit_score'].cumsum()
    sorted_r['population_pct'] = (np.arange(len(sorted_r)) + 1) / len(sorted_r)
    # By decile
    curve_rows = []
    for d in range(1, 11):
        subset = sorted_r[sorted_r['population_pct'] <= d / 10]
        curve_rows.append({'uplift_decile': d, 'cumulative_benefit': subset['benefit_score'].sum(),
            'population_fraction': d / 10})
    curve_df = pd.DataFrame(curve_rows)
    curve_df.to_csv(folder / 'uplift_curve_by_decile.csv', index=False)
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.plot(curve_df['population_fraction'], curve_df['cumulative_benefit'], marker='o', color='#4c78a8')
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.set_title(f'{model_label}: Uplift Curve (Cumulative Benefit)')
    ax.set_xlabel('Population Fraction (sorted by benefit score)')
    ax.set_ylabel('Cumulative Benefit Score')
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_uplift_curve_by_decile.png', dpi=150)
    plt.close(fig)
    # Summary
    pd.DataFrame([{'model': model_label, 'total_benefit': sorted_r['benefit_score'].sum(),
        'mean_benefit': sorted_r['benefit_score'].mean(),
        'top_decile_mean_benefit': sorted_r.head(len(sorted_r)//10)['benefit_score'].mean()}]).to_csv(
        folder / 'uplift_curve_summary.csv', index=False)


def save_observed_gap(results, folder, model_label):
    gap_rows = []
    for d in sorted(results['uplift_decile'].unique()):
        sub = results[results['uplift_decile'] == d]
        t_rate = sub.loc[sub['intervention_flag'] == 1, 'outcome_ed_90d'].mean()
        c_rate = sub.loc[sub['intervention_flag'] == 0, 'outcome_ed_90d'].mean()
        gap_rows.append({'uplift_decile': d, 'treated_ed_rate': t_rate, 'control_ed_rate': c_rate, 'observed_gap': c_rate - t_rate if pd.notna(c_rate) and pd.notna(t_rate) else np.nan})
    gap_df = pd.DataFrame(gap_rows)
    gap_df.to_csv(folder / 'uplift_observed_gap_by_decile.csv', index=False)
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.bar(gap_df['uplift_decile'].astype(str), gap_df['observed_gap'].fillna(0), color='#59a14f')
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.set_title(f'{model_label}: Observed Treated vs Control Gap by Decile')
    ax.set_xlabel('Uplift Decile')
    ax.set_ylabel('Control ED Rate - Treated ED Rate')
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_observed_gap_by_decile.png', dpi=150)
    plt.close(fig)


save_decile_dashboard_charts(decile_summary_xgboost, xgboost_output_folder, 'XGBoost')
save_uplift_curve(results_test_xgboost, xgboost_output_folder, 'XGBoost')
save_observed_gap(results_test_xgboost, xgboost_output_folder, 'XGBoost')
print('XGBoost dashboard charts saved.')

if decile_summary_glmnet is not None:
    save_decile_dashboard_charts(decile_summary_glmnet, glmnet_output_folder, 'GLMNet')
    save_uplift_curve(results_test_glmnet, glmnet_output_folder, 'GLMNet')
    save_observed_gap(results_test_glmnet, glmnet_output_folder, 'GLMNet')
    print('GLMNet dashboard charts saved.')


---
## 19. ROI Per Decile
---


In [ ]:
cost_per_ed_visit = 1200
cost_per_intervention = 250


def build_roi_summary(decile_df):
    roi_df = decile_df.copy()
    roi_df['expected_ed_rate_reduction'] = roi_df['avg_benefit_score']
    roi_df['expected_ed_visits_avoided'] = roi_df['n'] * roi_df['expected_ed_rate_reduction']
    roi_df['gross_savings'] = roi_df['expected_ed_visits_avoided'] * cost_per_ed_visit
    roi_df['intervention_cost'] = roi_df['n'] * cost_per_intervention
    roi_df['net_savings'] = roi_df['gross_savings'] - roi_df['intervention_cost']
    roi_df['roi'] = roi_df['net_savings'] / roi_df['intervention_cost']
    return roi_df


def save_roi_outputs(decile_df, folder, model_label):
    roi_df = build_roi_summary(decile_df)
    print(f'{model_label} ROI summary:')
    display(roi_df)
    roi_df.to_csv(folder / 'uplift_roi_by_decile.csv', index=False)
    save_bar_chart(roi_df, 'uplift_decile', 'net_savings',
        f'{model_label}: Estimated Net Savings by Uplift Decile',
        'Uplift Decile', 'Estimated Net Savings',
        folder / 'dashboard_roi_net_savings_by_decile.png')
    return roi_df


roi_summary_xgboost = save_roi_outputs(decile_summary_xgboost, xgboost_output_folder, 'XGBoost')
roi_summary = roi_summary_xgboost

if decile_summary_glmnet is not None:
    roi_summary_glmnet = save_roi_outputs(decile_summary_glmnet, glmnet_output_folder, 'GLMNet')
else:
    roi_summary_glmnet = None


---
## 20. Top Benefit Decile Summary
---


In [ ]:
def top_benefit_decile_summary(results, roi_df, model_label):
    if results is None or roi_df is None:
        return None
    top_decile = results[results['uplift_decile'] == 1].copy()
    if top_decile.empty:
        return None
    treated = top_decile[top_decile['intervention_flag'] == 1]
    control = top_decile[top_decile['intervention_flag'] == 0]
    treated_rate = treated['outcome_ed_90d'].mean() if not treated.empty else np.nan
    control_rate = control['outcome_ed_90d'].mean() if not control.empty else np.nan
    top_roi = roi_df.loc[roi_df['uplift_decile'] == 1].iloc[0]
    return pd.DataFrame([{
        'model': model_label,
        'top_decile_n': len(top_decile),
        'top_decile_treated_n': len(treated),
        'top_decile_control_n': len(control),
        'top_decile_avg_predicted_benefit': top_decile['benefit_score'].mean(),
        'top_decile_observed_ed_rate': top_decile['outcome_ed_90d'].mean(),
        'top_decile_treated_observed_ed_rate': treated_rate,
        'top_decile_control_observed_ed_rate': control_rate,
        'top_decile_observed_control_minus_treated_gap': control_rate - treated_rate if pd.notna(control_rate) and pd.notna(treated_rate) else np.nan,
        'top_decile_treated_pct': top_decile['intervention_flag'].mean(),
        'top_decile_estimated_ed_visits_avoided': top_roi['expected_ed_visits_avoided'],
        'top_decile_gross_savings': top_roi['gross_savings'],
        'top_decile_net_savings': top_roi['net_savings'],
        'top_decile_roi': top_roi['roi'],
    }])


def save_top_decile_summary(results, roi_df, folder, model_label):
    summary = top_benefit_decile_summary(results, roi_df, model_label)
    if summary is None:
        return None
    summary.to_csv(folder / 'top_benefit_decile_summary.csv', index=False)
    print(f'{model_label} top benefit decile summary:')
    display(summary)
    return summary


save_top_decile_summary(results_test_xgboost, roi_summary_xgboost, xgboost_output_folder, 'XGBoost')
if results_test_glmnet is not None and roi_summary_glmnet is not None:
    save_top_decile_summary(results_test_glmnet, roi_summary_glmnet, glmnet_output_folder, 'GLMNet')


---
## 21. X-Learner Implementation
---

Uses the already-trained T-learner outcome models to impute individual treatment effects, then trains second-stage treatment-effect models on those imputed effects.


In [ ]:
xlearner_root_folder = ensure_output_folder(output_folder / 'X-Learner')
xlearner_xgboost_output_folder = ensure_output_folder(xlearner_root_folder / 'XGBoost')
xlearner_glmnet_output_folder = ensure_output_folder(xlearner_root_folder / 'GLMNet')


def fit_xgb_regression_model(x_matrix, y, model_label, seed=123):
    params = xgb_training_params(
        XGB_GPU_PARAMS,
        {'max_depth': 3, 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8,
         'min_child_weight': 1, 'objective': 'reg:squarederror'},
        eval_metric='rmse', seed=seed,
    )
    model = xgb.train(params=params, dtrain=make_dmatrix(x_matrix, np.asarray(y, dtype=float)),
                      num_boost_round=200, verbose_eval=False)
    assert_xgb_booster_uses_cuda(model, model_label)
    return model


class PrefitScaledRegressionPipeline:
    def __init__(self, scaler, model):
        self.named_steps = {'standardscaler': scaler, 'elasticnetcv': model}

    def predict(self, x_matrix):
        return self.named_steps['elasticnetcv'].predict(
            self.named_steps['standardscaler'].transform(x_matrix))


def fit_glmnet_regression_model(x_matrix, y, seed=123, prefit_scaler=None):
    y_array = np.asarray(y, dtype=float)
    folds = int(min(5, len(y_array)))
    if folds < 2:
        raise ValueError('Need at least two rows for GLMNet X-learner regression.')
    reg_model = ElasticNetCV(
        l1_ratio=np.round(np.arange(0.0, 1.01, 0.1), 1),
        alphas=np.logspace(-4, 2, 50), cv=folds, max_iter=10000, random_state=seed,
    )
    if prefit_scaler is None:
        pipeline = make_pipeline(StandardScaler(), reg_model)
        pipeline.fit(x_matrix, y_array)
    else:
        x_scaled = prefit_scaler.transform(x_matrix)
        reg_model.fit(x_scaled, y_array)
        pipeline = PrefitScaledRegressionPipeline(prefit_scaler, reg_model)
    return pipeline


def fit_propensity_model(x_matrix, treatment, seed=123):
    t_array = np.asarray(treatment, dtype=float)
    class_counts = pd.Series(t_array).value_counts()
    folds = int(min(5, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two treatment classes for propensity modeling.')
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    prop_model = LogisticRegressionCV(
        Cs=np.logspace(-4, 4, 30), cv=cv, penalty='elasticnet', solver='saga',
        l1_ratios=[0.5], scoring='roc_auc', max_iter=10000, random_state=seed, refit=True,
    )
    pipeline = make_pipeline(StandardScaler(), prop_model)
    pipeline.fit(x_matrix, t_array)
    return pipeline


def clipped_propensity(propensity_model, x_matrix, lower=0.05, upper=0.95):
    propensity = propensity_model.predict_proba(x_matrix)[:, 1]
    return np.clip(propensity, lower, upper)


def build_xlearner_results(base_df, pred_treated, pred_control, cate_score, propensity_score, learner_label):
    results = base_df.copy()
    results['pred_ed_if_treated'] = pred_treated
    results['pred_ed_if_control'] = pred_control
    results['t_learner_benefit_score'] = results['pred_ed_if_control'] - results['pred_ed_if_treated']
    results['benefit_score'] = cate_score
    results['xlearner_propensity_score'] = propensity_score
    results['uplift_bad_outcome'] = -results['benefit_score']
    results['uplift_decile'] = ntile_desc(results['benefit_score'], 10).to_numpy()
    results['learner_framework'] = 'X-Learner'
    results['model'] = learner_label
    return results


---
## 22. Shared Propensity Model
---


In [ ]:
x_train_all = pd.concat([x_treated, x_control], axis=0)
t_train_all = pd.concat([
    pd.Series(np.ones(len(x_treated)), index=x_treated.index),
    pd.Series(np.zeros(len(x_control)), index=x_control.index),
], axis=0)

propensity_model = fit_propensity_model(x_train_all, t_train_all)
propensity_train_all = clipped_propensity(propensity_model, x_train_all)
propensity_test = clipped_propensity(propensity_model, x_test)

shared_propensity_scores = pd.concat([
    pd.DataFrame({
        'member_id': pd.concat([train_treated['member_id'], train_control['member_id']], axis=0).to_numpy(),
        'split': 'train', 'propensity_score': propensity_train_all,
        'propensity_model': 'GLMNet elastic-net logistic', 'seed': 123,
    }),
    pd.DataFrame({
        'member_id': test_df['member_id'].to_numpy(),
        'split': 'test', 'propensity_score': propensity_test,
        'propensity_model': 'GLMNet elastic-net logistic', 'seed': 123,
    }),
], ignore_index=True).sort_values('member_id').reset_index(drop=True)

shared_propensity_path = xlearner_root_folder / 'shared_propensity_scores.csv'
shared_propensity_scores.to_csv(shared_propensity_path, index=False)
print('Shared propensity scores written to:', shared_propensity_path)
print(f'Propensity score range: [{propensity_test.min():.3f}, {propensity_test.max():.3f}]')


---
## 23. XGBoost X-Learner
---


In [ ]:
# Impute individual benefit effects
xgb_mu0_on_treated = model_control.predict(make_dmatrix(x_treated))
xgb_mu1_on_control = model_treated.predict(make_dmatrix(x_control))
xgb_effect_treated = xgb_mu0_on_treated - np.asarray(y_treated, dtype=float)
xgb_effect_control = np.asarray(y_control, dtype=float) - xgb_mu1_on_control

xgb_tau_treated_model = fit_xgb_regression_model(x_treated, xgb_effect_treated, 'XGBoost X-learner treated-effect model')
xgb_tau_control_model = fit_xgb_regression_model(x_control, xgb_effect_control, 'XGBoost X-learner control-effect model')

xgb_tau_treated_test = xgb_tau_treated_model.predict(make_dmatrix(x_test))
xgb_tau_control_test = xgb_tau_control_model.predict(make_dmatrix(x_test))
xgb_xlearner_benefit = propensity_test * xgb_tau_control_test + (1 - propensity_test) * xgb_tau_treated_test

results_test_xlearner_xgboost = build_xlearner_results(
    test_df, p_treated_xgboost, p_control_xgboost, xgb_xlearner_benefit, propensity_test, 'XGBoost',
)
decile_summary_xlearner_xgboost = summarize_uplift_deciles(results_test_xlearner_xgboost)
print('XGBoost X-learner decile summary:')
display(decile_summary_xlearner_xgboost)
print_highest_benefit(results_test_xlearner_xgboost, 'XGBoost X-Learner')

results_test_xlearner_xgboost.to_csv(xlearner_xgboost_output_folder / 'xlearner_scored_test_output.csv', index=False)
decile_summary_xlearner_xgboost.to_csv(xlearner_xgboost_output_folder / 'xlearner_decile_summary.csv', index=False)


---
## 24. GLMNet X-Learner
---


In [ ]:
results_test_xlearner_glmnet = None
decile_summary_xlearner_glmnet = None
glmnet_tau_treated_model = None
glmnet_tau_control_model = None
xlearner_effect_shared_scaler = None

if enet_treated is not None and enet_control is not None:
    glmnet_mu0_on_treated = enet_control['best_model'].predict_proba(x_treated)[:, 1]
    glmnet_mu1_on_control = enet_treated['best_model'].predict_proba(x_control)[:, 1]
    glmnet_effect_treated = glmnet_mu0_on_treated - np.asarray(y_treated, dtype=float)
    glmnet_effect_control = np.asarray(y_control, dtype=float) - glmnet_mu1_on_control

    xlearner_effect_shared_scaler = StandardScaler().fit(pd.concat([x_treated, x_control], axis=0))
    glmnet_tau_treated_model = fit_glmnet_regression_model(x_treated, glmnet_effect_treated, prefit_scaler=xlearner_effect_shared_scaler)
    glmnet_tau_control_model = fit_glmnet_regression_model(x_control, glmnet_effect_control, prefit_scaler=xlearner_effect_shared_scaler)

    glmnet_tau_treated_test = glmnet_tau_treated_model.predict(x_test)
    glmnet_tau_control_test = glmnet_tau_control_model.predict(x_test)
    glmnet_xlearner_benefit = propensity_test * glmnet_tau_control_test + (1 - propensity_test) * glmnet_tau_treated_test

    results_test_xlearner_glmnet = build_xlearner_results(
        test_df, p_treated_glmnet, p_control_glmnet, glmnet_xlearner_benefit, propensity_test, 'GLMNet',
    )
    decile_summary_xlearner_glmnet = summarize_uplift_deciles(results_test_xlearner_glmnet)
    print('GLMNet X-learner decile summary:')
    display(decile_summary_xlearner_glmnet)
    print_highest_benefit(results_test_xlearner_glmnet, 'GLMNet X-Learner')

    results_test_xlearner_glmnet.to_csv(xlearner_glmnet_output_folder / 'xlearner_scored_test_output.csv', index=False)
    decile_summary_xlearner_glmnet.to_csv(xlearner_glmnet_output_folder / 'xlearner_decile_summary.csv', index=False)
else:
    print('Skipped GLMNet X-learner because GLMNet T-learner models were not available.')


---
## 25. X-Learner vs T-Learner Consistency
---


In [ ]:
def xlearner_consistency_summary(t_results, x_results, model_label):
    merged = pd.DataFrame({
        't_learner_benefit_score': t_results['benefit_score'].to_numpy(),
        'x_learner_benefit_score': x_results['benefit_score'].to_numpy(),
        't_learner_decile': t_results['uplift_decile'].to_numpy(),
        'x_learner_decile': x_results['uplift_decile'].to_numpy(),
    }, index=t_results.index)
    t_top = set(merged.index[merged['t_learner_decile'] == 1])
    x_top = set(merged.index[merged['x_learner_decile'] == 1])
    top_overlap = len(t_top & x_top) / len(t_top) if t_top else np.nan
    return pd.DataFrame([{
        'model': model_label,
        'pearson_benefit_score_corr': merged['t_learner_benefit_score'].corr(merged['x_learner_benefit_score'], method='pearson'),
        'spearman_benefit_score_corr': merged['t_learner_benefit_score'].corr(merged['x_learner_benefit_score'], method='spearman'),
        'top_decile_overlap_pct': top_overlap,
        't_learner_mean_benefit_score': merged['t_learner_benefit_score'].mean(),
        'x_learner_mean_benefit_score': merged['x_learner_benefit_score'].mean(),
    }])


consistency_frames = [xlearner_consistency_summary(results_test_xgboost, results_test_xlearner_xgboost, 'XGBoost')]
if results_test_glmnet is not None and results_test_xlearner_glmnet is not None:
    consistency_frames.append(xlearner_consistency_summary(results_test_glmnet, results_test_xlearner_glmnet, 'GLMNet'))

xlearner_consistency = pd.concat(consistency_frames, ignore_index=True)
xlearner_consistency.to_csv(xlearner_root_folder / 'xlearner_vs_tlearner_consistency_summary.csv', index=False)
print('X-learner vs T-learner consistency:')
display(xlearner_consistency)


---
## 26. Risk Tier Analysis & Stacked Bar Chart
---


In [ ]:
risk_tier_order = ['Low', 'Medium', 'High', 'Very High']
benefit_group_order = ['High benefit', 'Medium benefit', 'Low benefit']
benefit_group_colors = {'High benefit': '#1f77b4', 'Medium benefit': '#aec7e8', 'Low benefit': '#ffbb78'}

# Assign risk tiers based on current_risk_score quartiles
if 'current_risk_score' in model_df.columns and 'risk_tier' not in model_df.columns:
    q = model_df['current_risk_score'].quantile([0.25, 0.5, 0.75])
    model_df['risk_tier'] = pd.cut(model_df['current_risk_score'],
        bins=[-np.inf, q[0.25], q[0.5], q[0.75], np.inf],
        labels=risk_tier_order)

# Update test_df and results with risk_tier from model_df
if 'risk_tier' in model_df.columns:
    test_df_risk = model_df.loc[test_df.index, 'risk_tier'] if test_df.index.isin(model_df.index).all() else model_df.set_index('member_id').loc[test_df['member_id'], 'risk_tier'].values
    results_test_xgboost['risk_tier'] = test_df_risk if not isinstance(test_df_risk, pd.Series) else test_df_risk.values
    results_test_xgboost['current_risk_score'] = test_df['current_risk_score'].values if 'current_risk_score' in test_df.columns else np.nan
    if results_test_glmnet is not None:
        results_test_glmnet['risk_tier'] = results_test_xgboost['risk_tier'].values
        results_test_glmnet['current_risk_score'] = results_test_xgboost['current_risk_score'].values
    if results_test_xlearner_xgboost is not None:
        results_test_xlearner_xgboost['risk_tier'] = results_test_xgboost['risk_tier'].values
        results_test_xlearner_xgboost['current_risk_score'] = results_test_xgboost['current_risk_score'].values
    if results_test_xlearner_glmnet is not None:
        results_test_xlearner_glmnet['risk_tier'] = results_test_xgboost['risk_tier'].values
        results_test_xlearner_glmnet['current_risk_score'] = results_test_xgboost['current_risk_score'].values

# Risk tier thresholds
if 'current_risk_score' in model_df.columns:
    q = model_df['current_risk_score'].quantile([0.25, 0.5, 0.75])
    risk_tier_thresholds = pd.DataFrame([{
        'tier': t, 'lower': lb, 'upper': ub
    } for t, lb, ub in zip(risk_tier_order,
        [model_df['current_risk_score'].min(), q[0.25], q[0.5], q[0.75]],
        [q[0.25], q[0.5], q[0.75], model_df['current_risk_score'].max()])])
    risk_tier_thresholds.to_csv(output_folder / 'risk_tier_thresholds.csv', index=False)
    print('Risk tier thresholds:')
    display(risk_tier_thresholds)


In [ ]:
def assign_model_relative_benefit_group(uplift_decile):
    uplift_decile = int(uplift_decile)
    if uplift_decile in [1, 2]:
        return 'High benefit'
    if uplift_decile in [3, 4, 5, 6, 7]:
        return 'Medium benefit'
    return 'Low benefit'


def build_risk_tier_benefit_group_outputs(scored_df, output_dir, file_prefix, chart_title):
    df_local = scored_df.copy()
    if 'risk_tier' not in df_local.columns:
        print(f'Skipped {file_prefix}: risk_tier column not available.')
        return None
    df_local['risk_tier'] = pd.Categorical(df_local['risk_tier'], categories=risk_tier_order, ordered=True)
    df_local['benefit_group'] = df_local['uplift_decile'].apply(assign_model_relative_benefit_group)
    df_local['benefit_group'] = pd.Categorical(df_local['benefit_group'], categories=benefit_group_order, ordered=True)

    summary = df_local.groupby(['risk_tier', 'benefit_group'], observed=False).size().reset_index(name='members')
    tier_totals = df_local.groupby('risk_tier', observed=False).size().rename('risk_tier_members').reset_index()
    summary = summary.merge(tier_totals, on='risk_tier', how='left')
    summary['pct_within_risk_tier'] = np.where(summary['risk_tier_members'] > 0, summary['members'] / summary['risk_tier_members'], np.nan)
    summary.to_csv(output_dir / f'{file_prefix}_risk_tier_benefit_group_summary.csv', index=False)

    # Stacked bar chart
    pct = summary.pivot(index='risk_tier', columns='benefit_group', values='pct_within_risk_tier').reindex(risk_tier_order).fillna(0)
    counts = tier_totals.set_index('risk_tier').reindex(risk_tier_order)['risk_tier_members'].fillna(0).astype(int)
    fig, ax = plt.subplots(figsize=(9.2, 6.0))
    bottom = np.zeros(len(pct))
    x = np.arange(len(pct.index))
    for group in benefit_group_order:
        values = pct[group].to_numpy() if group in pct else np.zeros(len(pct))
        ax.bar(x, values, bottom=bottom, label=group, color=benefit_group_colors[group], edgecolor='white', linewidth=0.8)
        for idx, val in enumerate(values):
            if val >= 0.07:
                lc = 'white' if group == 'High benefit' else '#222222'
                ax.text(idx, bottom[idx] + val / 2, f'{val:.0%}', ha='center', va='center', fontsize=9, color=lc)
        bottom += values
    ax.set_xticks(x)
    ax.set_xticklabels([f'{t} risk\n(n={counts.loc[t]})' for t in risk_tier_order])
    ax.set_ylim(0, 1)
    ax.set_xlabel('Risk tier')
    ax.set_ylabel('Percent of members')
    ax.set_title(chart_title)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)
    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    fig.savefig(output_dir / f'dashboard_{file_prefix}_risk_tier_by_benefit_group.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    return summary


if results_test_glmnet is not None and 'risk_tier' in results_test_glmnet.columns:
    tlearner_rtbg = build_risk_tier_benefit_group_outputs(
        results_test_glmnet, glmnet_output_folder, 'tlearner',
        'GLMNet T-learner benefit group by risk tier (test set)')
    print('T-learner risk tier summary saved.')

if results_test_xlearner_glmnet is not None and 'risk_tier' in results_test_xlearner_glmnet.columns:
    xlearner_rtbg = build_risk_tier_benefit_group_outputs(
        results_test_xlearner_glmnet, xlearner_glmnet_output_folder, 'xlearner',
        'GLMNet X-learner benefit group by risk tier (test set)')
    print('X-learner risk tier summary saved.')

if results_test_xlearner_xgboost is not None and 'risk_tier' in results_test_xlearner_xgboost.columns:
    xgb_xlearner_rtbg = build_risk_tier_benefit_group_outputs(
        results_test_xlearner_xgboost, xlearner_xgboost_output_folder, 'xlearner',
        'XGBoost X-learner benefit group by risk tier (test set)')
    print('XGBoost X-learner risk tier summary saved.')


---
## 27. XGBoost X-Learner Feature Parity Outputs
---


In [ ]:
# XGBoost X-Learner decile dashboard
save_bar_chart(decile_summary_xlearner_xgboost, 'uplift_decile', 'avg_benefit_score',
    'XGBoost X-Learner: Average Predicted Benefit by Decile',
    'Uplift Decile', 'Average Benefit Score',
    xlearner_xgboost_output_folder / 'dashboard_avg_benefit_by_decile.png')

# XGBoost X-Learner ROI
xgb_xlearner_roi = build_roi_summary(decile_summary_xlearner_xgboost)
xgb_xlearner_roi.to_csv(xlearner_xgboost_output_folder / 'xlearner_roi_by_decile.csv', index=False)
save_bar_chart(xgb_xlearner_roi, 'uplift_decile', 'net_savings',
    'XGBoost X-Learner: Estimated Net Savings by Decile',
    'Uplift Decile', 'Estimated Net Savings',
    xlearner_xgboost_output_folder / 'dashboard_xlearner_roi_net_savings_by_decile.png')

# GLMNet X-Learner decile dashboard
if decile_summary_xlearner_glmnet is not None:
    save_bar_chart(decile_summary_xlearner_glmnet, 'uplift_decile', 'avg_benefit_score',
        'GLMNet X-Learner: Average Predicted Benefit by Decile',
        'Uplift Decile', 'Average Benefit Score',
        xlearner_glmnet_output_folder / 'dashboard_avg_benefit_by_decile.png')
    glmnet_xlearner_roi = build_roi_summary(decile_summary_xlearner_glmnet)
    glmnet_xlearner_roi.to_csv(xlearner_glmnet_output_folder / 'xlearner_roi_by_decile.csv', index=False)

print('X-Learner parity outputs complete.')


---
## 28. XGBoost X-Learner Benefit SHAP
---


In [ ]:
def generate_xlearner_benefit_shap_xgboost(tau_t_model, tau_c_model, prop_scores, x_matrix, output_dir, model_label):
    dmatrix = make_dmatrix(x_matrix)
    tau_t_contribs = tau_t_model.predict(dmatrix, pred_contribs=True)
    tau_c_contribs = tau_c_model.predict(dmatrix, pred_contribs=True)
    shap_cols = [*x_matrix.columns, 'BIAS']
    tau_t_shap = pd.DataFrame(tau_t_contribs, columns=shap_cols, index=x_matrix.index)
    tau_c_shap = pd.DataFrame(tau_c_contribs, columns=shap_cols, index=x_matrix.index)
    treated_weight = 1 - prop_scores
    control_weight = prop_scores
    weighted_t_shap = tau_t_shap.mul(treated_weight, axis=0)
    weighted_c_shap = tau_c_shap.mul(control_weight, axis=0)
    benefit_shap = weighted_t_shap + weighted_c_shap
    benefit_shap_no_bias = benefit_shap.drop(columns=['BIAS'], errors='ignore')
    importance_df = pd.DataFrame({
        'feature': benefit_shap_no_bias.columns,
        'mean_abs_benefit_shap': benefit_shap_no_bias.abs().mean(axis=0).to_numpy(),
        'mean_signed_benefit_shap': benefit_shap_no_bias.mean(axis=0).to_numpy(),
        'pct_positive_benefit_shap': benefit_shap_no_bias.gt(0).mean(axis=0).to_numpy(),
        'importance_type': 'xgboost_xlearner_weighted_treeshap',
    }).sort_values('mean_abs_benefit_shap', ascending=False).reset_index(drop=True)
    importance_df.to_csv(output_dir / 'shap_importance_benefit_score.csv', index=False)
    importance_df.to_csv(output_dir / 'xlearner_benefit_driver_importance.csv', index=False)
    top = importance_df.head(20).sort_values('mean_abs_benefit_shap')
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top['mean_abs_benefit_shap'])
    ax.set_title(f'{model_label} X-Learner: Top Benefit Drivers (TreeSHAP)')
    ax.set_xlabel('Mean Absolute SHAP Contribution to Benefit')
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(output_dir / 'dashboard_shap_benefit_score.png', dpi=150)
    fig.savefig(output_dir / 'dashboard_xlearner_benefit_drivers.png', dpi=150)
    plt.close(fig)
    print(f'{model_label} X-Learner TreeSHAP benefit-driver outputs saved.')
    return importance_df


xgb_xlearner_shap = generate_xlearner_benefit_shap_xgboost(
    xgb_tau_treated_model, xgb_tau_control_model, propensity_test, x_test,
    xlearner_xgboost_output_folder, 'XGBoost')
print('Top XGBoost X-Learner benefit SHAP features:')
display(xgb_xlearner_shap.head(15))


---
## 29. Cumulative & Marginal Targeting Savings
---


In [ ]:
def generate_xlearner_cumulative_targeting(results_df, output_dir, model_label):
    n_total = len(results_df)
    if n_total == 0:
        return None, None, None
    decile_size = max(1, n_total // 10)
    ranking_specs = [('Uplift score', 'benefit_score', False)]
    if 'current_risk_score' in results_df.columns:
        ranking_specs.append(('Current risk score', 'current_risk_score', False))
    rows = []
    for approach, sort_col, ascending in ranking_specs:
        ranked = results_df.sort_values(sort_col, ascending=ascending).reset_index(drop=True)
        for decile in range(1, 11):
            top_n = n_total if decile == 10 else decile * decile_size
            selected = ranked.head(top_n)
            avoided = float(selected['benefit_score'].sum())
            rows.append({'targeting_approach': approach, 'through_decile': decile,
                'population_fraction_targeted': top_n / n_total, 'n': top_n,
                'cumulative_estimated_ed_visits_avoided': avoided,
                'cumulative_gross_savings': avoided * cost_per_ed_visit})
    cumulative_df = pd.DataFrame(rows)
    cumulative_df.to_csv(output_dir / 'cumulative_gross_savings_by_targeting.csv', index=False)

    # Marginal
    marginal_rows = []
    for approach in cumulative_df['targeting_approach'].unique():
        subset = cumulative_df[cumulative_df['targeting_approach'] == approach].sort_values('through_decile')
        prev = 0.0
        for _, row in subset.iterrows():
            marginal = row['cumulative_gross_savings'] - prev
            marginal_rows.append({'targeting_approach': approach, 'decile': int(row['through_decile']),
                'marginal_gross_savings': marginal, 'cumulative_gross_savings': row['cumulative_gross_savings']})
            prev = row['cumulative_gross_savings']
    marginal_df = pd.DataFrame(marginal_rows)
    marginal_df.to_csv(output_dir / 'marginal_gross_savings_by_targeting.csv', index=False)

    # Chart
    chart_df = cumulative_df[cumulative_df['population_fraction_targeted'] <= 0.51]
    fig, ax = plt.subplots(figsize=(8.5, 5.25))
    for approach in chart_df['targeting_approach'].unique():
        adf = chart_df[chart_df['targeting_approach'] == approach]
        ax.plot(adf['population_fraction_targeted'], adf['cumulative_gross_savings'], marker='o', label=approach)
    ax.set_title(f'{model_label} X-Learner: Cumulative Gross Savings by Targeting')
    ax.set_xlabel('Population Fraction Targeted')
    ax.set_ylabel('Cumulative Gross Savings ($)')
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_dir / 'dashboard_cumulative_gross_savings_targeting.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'{model_label} X-Learner targeting savings saved to:', output_dir)
    return cumulative_df, marginal_df


generate_xlearner_cumulative_targeting(results_test_xlearner_xgboost, xlearner_xgboost_output_folder, 'XGBoost')
if results_test_xlearner_glmnet is not None:
    generate_xlearner_cumulative_targeting(results_test_xlearner_glmnet, xlearner_glmnet_output_folder, 'GLMNet')


---
## 30. Predictor Data Dictionary & Distribution Summaries
---


In [ ]:
predictor_category_lookup = {
    'client_contract': 'Demographics', 'service_region': 'Demographics', 'program': 'Demographics',
    'case_manager_name': 'Demographics', 'age': 'Demographics', 'gender': 'Demographics',
    'dual_eligible': 'Demographics', 'county': 'Demographics', 'plan_type': 'Demographics',
    'language': 'Demographics', 'living_alone_flag': 'Demographics',
    'diabetes_flag': 'Clinical Conditions', 'chf_flag': 'Clinical Conditions',
    'copd_flag': 'Clinical Conditions', 'asthma_flag': 'Clinical Conditions',
    'depression_flag': 'Clinical Conditions', 'anxiety_flag': 'Clinical Conditions',
    'substance_use_flag': 'Clinical Conditions', 'ckd_flag': 'Clinical Conditions',
    'pregnancy_flag': 'Clinical Conditions', 'behavioral_health_risk_flag': 'Clinical Conditions',
    'food_insecurity_flag': 'SDOH', 'housing_instability_flag': 'SDOH',
    'transportation_barrier_flag': 'SDOH', 'utilities_insecurity_flag': 'SDOH',
    'pcp_visits_last_6m': 'Utilization', 'specialist_visits_last_6m': 'Utilization',
    'ed_visits_last_30d': 'Utilization', 'ed_visits_last_6m': 'Utilization',
    'admits_last_6m': 'Utilization', 'observation_stays_last_6m': 'Utilization',
    'total_cost_last_6m': 'Pharmacy', 'rx_count_last_6m': 'Pharmacy',
    'med_adherence_pdc': 'Pharmacy', 'high_cost_drug_flag': 'Pharmacy',
    'opioid_flag': 'Pharmacy', 'polypharmacy_flag': 'Pharmacy',
    'percolator_utilization_score': 'Risk Scores', 'percolator_clinical_score': 'Risk Scores',
    'percolator_sdoh_score': 'Risk Scores', 'current_risk_score': 'Risk Scores', 'risk_tier': 'Risk Scores',
}

predictor_description_lookup = {
    'age': 'Member age at the index point.',
    'gender': 'Member gender category.',
    'dual_eligible': 'Medicare/Medicaid dual eligibility indicator.',
    'diabetes_flag': 'Diabetes diagnosis/history indicator.',
    'chf_flag': 'Congestive heart failure indicator.',
    'copd_flag': 'COPD diagnosis indicator.',
    'ed_visits_last_6m': 'ED visits in last 6 months.',
    'total_cost_last_6m': 'Total healthcare cost in last 6 months.',
    'current_risk_score': 'Current overall risk score.',
    'percolator_clinical_score': 'Composite clinical risk score.',
    'percolator_sdoh_score': 'Composite SDOH risk score.',
    'percolator_utilization_score': 'Composite utilization risk score.',
}


def predictor_category(col):
    return predictor_category_lookup.get(col, 'Other / derived')


def predictor_description(col):
    return predictor_description_lookup.get(col, f'Model predictor derived from `{col}`.')


predictor_dict_rows = []
for col in candidate_predictors:
    raw_series = df[col] if col in df.columns else pd.Series(dtype='object')
    model_series = model_df[col] if col in model_df.columns else raw_series
    numeric_values = pd.to_numeric(raw_series, errors='coerce')
    predictor_dict_rows.append({
        'variable': col,
        'category': predictor_category(col),
        'description': predictor_description(col),
        'included_in_model': col in feature_cols,
        'missing_count': int(raw_series.isna().sum()),
        'missing_pct': float(raw_series.isna().mean()),
        'unique_values': int(raw_series.dropna().nunique()),
        'min_value': float(numeric_values.min()) if numeric_values.notna().any() else np.nan,
        'max_value': float(numeric_values.max()) if numeric_values.notna().any() else np.nan,
    })

predictor_data_dictionary = pd.DataFrame(predictor_dict_rows).sort_values(['category', 'variable']).reset_index(drop=True)
predictor_data_dictionary.to_csv(predictor_dist_folder / 'predictor_data_dictionary.csv', index=False)
print('Predictor data dictionary:')
display(predictor_data_dictionary)


---
## 31. Predictor Distribution Visuals
---


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import re as _re

numeric_distribution_folder = ensure_output_folder(predictor_dist_folder / 'Numeric')
categorical_distribution_folder = ensure_output_folder(predictor_dist_folder / 'Categorical')


def safe_filename(val):
    return _re.sub(r'[^A-Za-z0-9_.-]+', '_', str(val)).strip('_') or 'unnamed'


# Numeric summary
numeric_summary_rows = []
categorical_summary_rows = []
for col in candidate_predictors:
    raw_series = df[col] if col in df.columns else pd.Series(dtype='object')
    numeric_values = pd.to_numeric(raw_series, errors='coerce')
    is_num = col in possible_numeric_cols and not is_binary_indicator_column(raw_series) and numeric_values.notna().any()
    if is_num:
        numeric_summary_rows.append({
            'variable': col, 'category': predictor_category(col),
            'n': int(numeric_values.notna().sum()),
            'mean': float(numeric_values.mean()), 'median': float(numeric_values.median()),
            'std': float(numeric_values.std()),
            'min': float(numeric_values.min()), 'max': float(numeric_values.max()),
        })
    else:
        non_missing = raw_series.dropna().astype(str)
        vc = non_missing.value_counts()
        categorical_summary_rows.append({
            'variable': col, 'category': predictor_category(col),
            'n': int(non_missing.shape[0]),
            'unique_values': int(non_missing.nunique()),
            'mode': vc.index[0] if len(vc) else np.nan,
            'mode_count': int(vc.iloc[0]) if len(vc) else 0,
        })

numeric_predictor_summary = pd.DataFrame(numeric_summary_rows).sort_values(['category', 'variable']).reset_index(drop=True)
categorical_predictor_summary = pd.DataFrame(categorical_summary_rows).sort_values(['category', 'variable']).reset_index(drop=True)
numeric_predictor_summary.to_csv(predictor_dist_folder / 'numeric_predictor_summary.csv', index=False)
categorical_predictor_summary.to_csv(predictor_dist_folder / 'categorical_predictor_summary.csv', index=False)

# Generate distribution PDFs
numeric_pdf_path = predictor_dist_folder / 'numeric_predictor_distributions.pdf'
categorical_pdf_path = predictor_dist_folder / 'categorical_predictor_distributions.pdf'

with PdfPages(numeric_pdf_path) as pdf:
    for col in numeric_predictor_summary['variable'].tolist():
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if values.empty:
            continue
        fig, ax = plt.subplots(figsize=(8, 4.8))
        if values.nunique() <= 15:
            sns.histplot(values, discrete=True, shrink=0.8, ax=ax, color='#4c78a8')
        else:
            sns.histplot(values, bins=30, kde=True, ax=ax, color='#4c78a8')
        ax.set_title(f'{col} Distribution')
        ax.set_xlabel(col)
        ax.set_ylabel('Member count')
        fig.tight_layout()
        fig.savefig(numeric_distribution_folder / f'{safe_filename(col)}_histogram.png', dpi=150, bbox_inches='tight')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

with PdfPages(categorical_pdf_path) as pdf:
    for col in categorical_predictor_summary['variable'].tolist():
        values = df[col].dropna().astype(str) if col in df.columns else pd.Series(dtype=str)
        if values.empty:
            continue
        counts = values.value_counts()
        if len(counts) > 20:
            counts = pd.concat([counts.head(20), pd.Series({'Other': counts.iloc[20:].sum()})])
        fig_h = max(4.8, min(12, 0.35 * len(counts) + 2.2))
        fig, ax = plt.subplots(figsize=(9, fig_h))
        plot_df = counts.reset_index()
        plot_df.columns = [col, 'count']
        sns.barplot(data=plot_df, x='count', y=col, ax=ax, color='#59a14f')
        ax.set_title(f'{col} Distribution')
        fig.tight_layout()
        fig.savefig(categorical_distribution_folder / f'{safe_filename(col)}_bar_chart.png', dpi=150, bbox_inches='tight')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print('Predictor distribution visuals saved to:', predictor_dist_folder)
print('Numeric PDF:', numeric_pdf_path)
print('Categorical PDF:', categorical_pdf_path)


---
## 32. Model Evaluation Summary
---


In [ ]:
eval_rows = []

# XGBoost T-Learner
eval_rows.append({
    'model': 'XGBoost T-Learner',
    'treated_cv_auc': xgb_treated_cv['best_cv_auc'],
    'control_cv_auc': xgb_control_cv['best_cv_auc'],
    'treated_test_auc': auc_treated_cv_xgb,
    'control_test_auc': auc_control_cv_xgb,
    'mean_benefit_score': results_test_xgboost['benefit_score'].mean(),
    'top_decile_mean_benefit': decile_summary_xgboost.loc[decile_summary_xgboost['uplift_decile'] == 1, 'avg_benefit_score'].iloc[0],
})

# GLMNet T-Learner
if enet_treated is not None and enet_control is not None:
    eval_rows.append({
        'model': 'GLMNet T-Learner',
        'treated_cv_auc': enet_treated['best_auc'],
        'control_cv_auc': enet_control['best_auc'],
        'treated_test_auc': safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], p_treated_glmnet[test_treated_pos], 'GLMNet Treated test') if p_treated_glmnet is not None else np.nan,
        'control_test_auc': safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], p_control_glmnet[test_control_pos], 'GLMNet Control test') if p_control_glmnet is not None else np.nan,
        'mean_benefit_score': results_test_glmnet['benefit_score'].mean() if results_test_glmnet is not None else np.nan,
        'top_decile_mean_benefit': decile_summary_glmnet.loc[decile_summary_glmnet['uplift_decile'] == 1, 'avg_benefit_score'].iloc[0] if decile_summary_glmnet is not None else np.nan,
    })

model_evaluation_summary = pd.DataFrame(eval_rows)
model_evaluation_summary.to_csv(output_folder / 'model_evaluation_summary.csv', index=False)
print('Model evaluation summary:')
display(model_evaluation_summary)


---
## 33. Factual Outcome Model Diagnostics
---


In [ ]:
def factual_event_count_summary(train_frame, test_frame):
    rows = []
    for split_name, frame in [('Train', train_frame), ('Test', test_frame)]:
        for group_label, tv in [('Treated', 1), ('Control', 0)]:
            sub = frame[frame['intervention_flag'] == tv]
            n = len(sub)
            events = int(sub['outcome_ed_90d'].sum()) if n else 0
            rows.append({'split': split_name, 'group': group_label, 'n': n,
                'positive_ed_events': events, 'negative_ed_events': n - events,
                'event_rate': events / n if n else np.nan})
    return pd.DataFrame(rows)


event_count_summary = factual_event_count_summary(train_df, test_df)
event_count_summary.to_csv(output_folder / 'factual_event_count_summary.csv', index=False)
print('Factual event count summary:')
display(event_count_summary)

# Factual prediction separation diagnostics
def factual_prediction_separation(results, model_label):
    rows = []
    for group_label, tv, pred_col in [('Treated', 1, 'pred_ed_if_treated'), ('Control', 0, 'pred_ed_if_control')]:
        sub = results[results['intervention_flag'] == tv].copy()
        if sub.empty:
            continue
        pos = sub[sub['outcome_ed_90d'] == 1]
        neg = sub[sub['outcome_ed_90d'] == 0]
        auc_val = roc_auc_score(sub['outcome_ed_90d'], sub[pred_col]) if sub['outcome_ed_90d'].nunique() == 2 else np.nan
        rows.append({'model': model_label, 'group': group_label, 'n': len(sub),
            'auc': auc_val,
            'avg_pred_actual_positive': pos[pred_col].mean() if len(pos) else np.nan,
            'avg_pred_actual_negative': neg[pred_col].mean() if len(neg) else np.nan,
            'avg_pred_positive_minus_negative': (pos[pred_col].mean() - neg[pred_col].mean()) if len(pos) and len(neg) else np.nan,
        })
    return pd.DataFrame(rows)


sep_frames = [factual_prediction_separation(results_test_xgboost, 'XGBoost')]
if results_test_glmnet is not None:
    sep_frames.append(factual_prediction_separation(results_test_glmnet, 'GLMNet'))
factual_sep = pd.concat(sep_frames, ignore_index=True)
factual_sep.to_csv(output_folder / 'factual_prediction_separation.csv', index=False)
print('Factual prediction separation:')
display(factual_sep)


---
## 34. GenRocket Ground-Truth Validation
---

The GenRocket dataset includes `true_treatment_effect` and `Propensity_Score` columns that enable validation of the uplift model against known ground truth.


In [ ]:
# Validate against ground truth if available
if 'true_treatment_effect' in raw.columns:
    # Map true treatment effect to test set members
    true_effect_test = raw.iloc[test_df.index]['true_treatment_effect'].values if len(raw) == len(model_df) else np.nan

    if not np.all(np.isnan(true_effect_test)):
        # T-Learner XGBoost correlation with true effect
        xgb_benefit_test = results_test_xgboost['benefit_score'].values
        corr_pearson = pd.Series(xgb_benefit_test).corr(pd.Series(true_effect_test))
        corr_spearman = pd.Series(xgb_benefit_test).corr(pd.Series(true_effect_test), method='spearman')

        validation_summary = pd.DataFrame([{
            'model': 'XGBoost T-Learner',
            'pearson_corr_with_true_effect': corr_pearson,
            'spearman_corr_with_true_effect': corr_spearman,
            'mean_predicted_benefit': float(np.mean(xgb_benefit_test)),
            'mean_true_effect': float(np.nanmean(true_effect_test)),
        }])

        # X-Learner XGBoost
        xl_benefit_test = results_test_xlearner_xgboost['benefit_score'].values
        validation_summary = pd.concat([validation_summary, pd.DataFrame([{
            'model': 'XGBoost X-Learner',
            'pearson_corr_with_true_effect': pd.Series(xl_benefit_test).corr(pd.Series(true_effect_test)),
            'spearman_corr_with_true_effect': pd.Series(xl_benefit_test).corr(pd.Series(true_effect_test), method='spearman'),
            'mean_predicted_benefit': float(np.mean(xl_benefit_test)),
            'mean_true_effect': float(np.nanmean(true_effect_test)),
        }])], ignore_index=True)

        validation_summary.to_csv(output_folder / 'genrocket_true_effect_validation.csv', index=False)
        print('GenRocket ground-truth validation:')
        display(validation_summary)
    else:
        print('true_treatment_effect values are all NaN for test set — skipping validation.')
else:
    print('true_treatment_effect column not found in raw data — skipping validation.')


---
## 35. Risk Tier Population Summary
---


In [ ]:
if 'risk_tier' in model_df.columns and 'current_risk_score' in model_df.columns:
    risk_tier_population_summary = (
        model_df.groupby('risk_tier', observed=False)
        .agg(
            n=('outcome_ed_90d', 'size'),
            outcome_rate=('outcome_ed_90d', 'mean'),
            treatment_rate=('intervention_flag', 'mean'),
            avg_current_risk_score=('current_risk_score', 'mean'),
        )
        .reindex(risk_tier_order)
        .reset_index()
    )
    risk_tier_population_summary.to_csv(output_folder / 'risk_tier_population_summary.csv', index=False)
    print('Risk tier population summary:')
    display(risk_tier_population_summary)
else:
    print('Skipped risk tier population summary — columns not available.')


---
## 36. Model Recommendation Summary
---


In [ ]:
recommendation_rows = []
recommendation_rows.append({
    'model': 'XGBoost T-Learner',
    'framework': 'T-Learner',
    'engine': 'XGBoost (GPU)',
    'mean_benefit_score': results_test_xgboost['benefit_score'].mean(),
    'top_decile_benefit': decile_summary_xgboost.loc[decile_summary_xgboost['uplift_decile'] == 1, 'avg_benefit_score'].iloc[0],
    'top_decile_roi': roi_summary_xgboost.loc[roi_summary_xgboost['uplift_decile'] == 1, 'roi'].iloc[0],
})
recommendation_rows.append({
    'model': 'XGBoost X-Learner',
    'framework': 'X-Learner',
    'engine': 'XGBoost (GPU)',
    'mean_benefit_score': results_test_xlearner_xgboost['benefit_score'].mean(),
    'top_decile_benefit': decile_summary_xlearner_xgboost.loc[decile_summary_xlearner_xgboost['uplift_decile'] == 1, 'avg_benefit_score'].iloc[0],
    'top_decile_roi': xgb_xlearner_roi.loc[xgb_xlearner_roi['uplift_decile'] == 1, 'roi'].iloc[0],
})
if results_test_glmnet is not None:
    recommendation_rows.append({
        'model': 'GLMNet T-Learner',
        'framework': 'T-Learner',
        'engine': 'GLMNet (CPU)',
        'mean_benefit_score': results_test_glmnet['benefit_score'].mean(),
        'top_decile_benefit': decile_summary_glmnet.loc[decile_summary_glmnet['uplift_decile'] == 1, 'avg_benefit_score'].iloc[0],
        'top_decile_roi': roi_summary_glmnet.loc[roi_summary_glmnet['uplift_decile'] == 1, 'roi'].iloc[0] if roi_summary_glmnet is not None else np.nan,
    })
if results_test_xlearner_glmnet is not None:
    glmnet_xl_roi = build_roi_summary(decile_summary_xlearner_glmnet)
    recommendation_rows.append({
        'model': 'GLMNet X-Learner',
        'framework': 'X-Learner',
        'engine': 'GLMNet (CPU)',
        'mean_benefit_score': results_test_xlearner_glmnet['benefit_score'].mean(),
        'top_decile_benefit': decile_summary_xlearner_glmnet.loc[decile_summary_xlearner_glmnet['uplift_decile'] == 1, 'avg_benefit_score'].iloc[0],
        'top_decile_roi': glmnet_xl_roi.loc[glmnet_xl_roi['uplift_decile'] == 1, 'roi'].iloc[0],
    })

model_recommendation_summary = pd.DataFrame(recommendation_rows).sort_values('top_decile_benefit', ascending=False).reset_index(drop=True)
model_recommendation_summary.to_csv(output_folder / 'model_recommendation_summary.csv', index=False)
print('Model recommendation summary:')
display(model_recommendation_summary)


---
## 37. Output File Index
---


In [ ]:
writeup_output_files = [
    output_folder / 'data_review_summary.csv',
    output_folder / 'model_evaluation_summary.csv',
    output_folder / 'model_recommendation_summary.csv',
    output_folder / 'factual_event_count_summary.csv',
    output_folder / 'factual_prediction_separation.csv',
    output_folder / 'risk_tier_population_summary.csv',
    output_folder / 'risk_tier_thresholds.csv',
    predictor_dist_folder / 'predictor_data_dictionary.csv',
    predictor_dist_folder / 'numeric_predictor_summary.csv',
    predictor_dist_folder / 'categorical_predictor_summary.csv',
    predictor_dist_folder / 'numeric_predictor_distributions.pdf',
    predictor_dist_folder / 'categorical_predictor_distributions.pdf',
    xlearner_root_folder / 'xlearner_vs_tlearner_consistency_summary.csv',
    xlearner_root_folder / 'shared_propensity_scores.csv',
]

for folder in [xgboost_output_folder, glmnet_output_folder]:
    writeup_output_files.extend([
        folder / 'uplift_scored_output.csv',
        folder / 'uplift_decile_summary.csv',
        folder / 'uplift_roi_by_decile.csv',
        folder / 'model_brier_scores.csv',
        folder / 'calibration_summary.csv',
        folder / 'calibration_by_decile.csv',
        folder / 'uplift_observed_gap_by_decile.csv',
        folder / 'uplift_curve_by_decile.csv',
        folder / 'uplift_curve_summary.csv',
        folder / 'top_benefit_decile_summary.csv',
        folder / 'shap_importance_treated_control_models.csv',
        folder / 'shap_importance_benefit_score.csv',
        folder / 'dashboard_calibration_plot.png',
        folder / 'dashboard_observed_gap_by_decile.png',
        folder / 'dashboard_uplift_curve_by_decile.png',
        folder / 'dashboard_shap_benefit_score.png',
        folder / 'dashboard_avg_benefit_by_decile.png',
        folder / 'dashboard_roi_net_savings_by_decile.png',
    ])

for folder in [xlearner_xgboost_output_folder, xlearner_glmnet_output_folder]:
    writeup_output_files.extend([
        folder / 'xlearner_scored_test_output.csv',
        folder / 'xlearner_decile_summary.csv',
        folder / 'xlearner_roi_by_decile.csv',
        folder / 'dashboard_avg_benefit_by_decile.png',
        folder / 'shap_importance_benefit_score.csv',
        folder / 'cumulative_gross_savings_by_targeting.csv',
        folder / 'marginal_gross_savings_by_targeting.csv',
    ])

writeup_output_index = pd.DataFrame({
    'output_file': [str(p) for p in writeup_output_files],
    'exists': [p.exists() for p in writeup_output_files],
})

print(f'Output file index: {writeup_output_index["exists"].sum()} / {len(writeup_output_index)} files exist.')
display(writeup_output_index)


---
## 38. Generate README
---


In [ ]:
readme_lines = [
    '# PRISM Uplift Modeling — GenRocket 10k Dataset',
    '',
    '## Overview',
    '',
    'This folder contains outputs from the PRISM Uplift Modeling workflow applied to the GenRocket',
    'synthetic 10k dataset (`Genrocket_10k_seed1_updated.csv`).',
    '',
    '**Outcome:** `outcome_ed_90d` (binary: 90-day ED visit)',
    '**Treatment:** `intervention_flag` (binary: received intervention)',
    '',
    '## Models',
    '',
    '| Model | Framework | Engine |',
    '|-------|-----------|--------|',
    '| XGBoost T-Learner | T-Learner | XGBoost GPU |',
    '| XGBoost X-Learner | X-Learner | XGBoost GPU |',
    '| GLMNet T-Learner | T-Learner | sklearn CPU |',
    '| GLMNet X-Learner | X-Learner | sklearn CPU |',
    '',
    '## Folder Structure',
    '',
    '```',
    'Outputs/upliftGenrocket/Python/',
    '├── T-Learner/',
    '│   ├── XGBoost/    # XGBoost T-Learner scored output, deciles, ROI, SHAP',
    '│   └── GLMNet/     # GLMNet T-Learner scored output, deciles, ROI, contributions',
    '├── X-Learner/',
    '│   ├── XGBoost/    # XGBoost X-Learner scored output, deciles, benefit drivers',
    '│   └── GLMNet/     # GLMNet X-Learner scored output, deciles, benefit drivers',
    '├── Predictor_Distributions/',
    '│   ├── Numeric/    # Individual numeric predictor histogram PNGs',
    '│   └── Categorical/ # Individual categorical bar chart PNGs',
    '├── data_review_summary.csv',
    '├── model_evaluation_summary.csv',
    '├── model_recommendation_summary.csv',
    '├── risk_tier_population_summary.csv',
    '└── README.md',
    '```',
    '',
    '## Dataset',
    '',
    f'- **Source:** `DataSets/Genrocket_10k_seed1_updated.csv`',
    f'- **Rows:** {len(model_df):,}',
    f'- **Predictors:** {len(feature_cols)}',
    f'- **Model matrix columns:** {x_test.shape[1]}',
    f'- **Treatment rate:** {model_df["intervention_flag"].mean():.1%}',
    f'- **Outcome rate:** {model_df["outcome_ed_90d"].mean():.1%}',
    '',
    '## Notes',
    '',
    '- Binary flags in the GenRocket data use Y/N encoding (converted to 0/1 during preprocessing).',
    '- Column renames: `Clinical Score` → `percolator_clinical_score`, `SDOH Score` → `percolator_sdoh_score`,',
    '  `utilization_score` → `percolator_utilization_score`, `currentRiskScore` → `current_risk_score`.',
    '- The dataset includes `true_treatment_effect` and `Propensity_Score` for ground-truth validation.',
    '- Train/test split: 70/30, seed=123, stratified on intervention_flag + outcome_ed_90d.',
    '',
    f'Generated by: `Code/PRISM_Uplift_GenRocket_Modeling_Workflow.ipynb`',
]

readme_path = output_folder / 'README.md'
readme_path.write_text('\n'.join(readme_lines), encoding='utf-8')
print('README written to:', readme_path)


---
## Done
---

All outputs have been written to `Outputs/upliftGenrocket/Python/`.


In [ ]:
print('='*60)
print('PRISM Uplift Modeling Workflow — GenRocket 10k Dataset')
print('='*60)
print()
print(f'Total output files generated: {writeup_output_index["exists"].sum()}')
print(f'Output root: {output_folder}')
print()
print('Workflow complete.')
